In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import lucent
import matplotlib.pyplot as plt
from lucent.optvis import render, param, transform, objectives
from lucent.modelzoo import inceptionv1
from pathlib import Path
import torch
from lucent.optvis.objectives import wrap_objective, handle_batch
from torch.nn import functional as F
import numpy as np
from olt.tfms import transform, inverse_transform
from PIL import Image
from olt.act import InputOutputModelSnapshot
import torch
from olt.html_report import apply_cmap, to_pil, rd_bk_gn
from olt.shards import read_image_shard
import warnings
from tqdm import tqdm
from olt.feature_viz import tensor_to_img_array
from olt.feature_viz import get_feature_viz_input


device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


layer_name = "mixed4e_1x1_pre_relu_conv"
channel = 55

plt.style.use("dark_background")

In [ ]:
@wrap_objective()
def neg_channel(layer, n_channel, batch=None):
    """Visualize a single channel"""
    @handle_batch(batch)
    def inner(model):
        res = model(layer)[:, n_channel].mean()
        # print(res)
        return res
    return inner


@wrap_objective()
def mse_at_position(layer, n_channel, pos, value, batch=None):
    # todo: remove this, the fn below is more useful
    y, x = pos
    @handle_batch(batch)
    def inner(model):
        o = model(layer)[:, n_channel]
        cur_val = o[:, y, x]
        # print("cur", cur_val, "us", value)
        res = F.mse_loss(o[:, y, x], value)
        return res
    return inner

@wrap_objective()
def mse_at_multiple_positions(layer, n_channel, positions, values, batch=None): 
    @handle_batch(batch)
    def inner(model):
        o = model(layer)
        mse = 0.
        for pos, val in zip(positions, values):
            y, x = pos
            mse += F.mse_loss(o[:, n_channel, y, x], val)
        return mse
    return inner


@wrap_objective()
def exact_tensor(layer, n_channel, value, batch=None):
    @handle_batch(batch)
    def inner(model):
        o = model(layer)[:, n_channel]
        res = F.mse_loss(o, value)
        return res
    return inner


@wrap_objective()
def exact_tensor_all_chans(layer, value, batch=None):
    @handle_batch(batch)
    def inner(model):
        o = model(layer)
        res = F.mse_loss(o, value)
        return res
    return inner


def get_batch_from_feature_vis(viz):
    img = np.floor(viz * 256).astype(np.uint8)
    timg = transform(Image.fromarray(img))[None]
    return timg



def make_heatmap(model, layer_name, neuron_selector, pil_img):
    # pil_img = Image.open(FLAT_IMAGES_BASE / f"{input_key}.jpeg")
    # idx = indices[sorted_idxs[0]]
    timg = transform(pil_img)[None]
    inv_img = inverse_transform(timg)
    ndl = NeuronDeepLift(model, model.get_submodule(layer_name))
    attr_res = ndl.attribute(timg, neuron_selector)

    size = (224, 224)

    overlay = attr_res[0].detach().cpu().sum(dim=0).numpy()
    overlay = overlay / np.abs(overlay).max()

    base = to_pil(inv_img[0], size=size)
    heat = apply_cmap(
        overlay,
        rd_bk_gn,
        vmin=-1,
        vmax=1,
        size=size,
        interpolation=Image.BILINEAR,
    )
    cell = Image.blend(base, heat, alpha=0.8)
    return cell

@wrap_objective()
def patch_across_channel(layer, patches, positions, batch=None):
    @handle_batch(batch)
    def inner(model):
        o = model(layer)
        # print("orig out shape", o.shape)
        mse = 0
        for patch, pos in zip(patches, positions):
            y, x = pos
            mse += F.mse_loss(o[:, :, y, x], patch)
        return mse
    return inner


First we run feature visualisation for the standard maximising objective.  
Look at the activations, the max is also hovering around our snout range (many are quite lesser than it in fact)

# Get activation

What is the activation at 6,2? We have a (14,14) output, 528 vector.  


mixed 4d:

- mixed4d_1x1_pre_relu_conv: 112
- mixed4d_3x3_pre_relu_conv: 288
- mixed4d_5x5_pre_relu_conv: 64
- mixed4d_pool_reduce_pre_relu_conv: 64

= 528 chans

We just need to extract mixed4d, all chans, single location 


In [ ]:
import pandas as pd
from olt.tfms import transform, inverse_transform
from olt.act import InputOutputModelSnapshot, get_layer_activations
from olt.html_report import make_overlay_heatmap


In [ ]:
report_csv = Path.home() / "Downloads/lucent-reports/overlay-reports-hyperparameter-grid-search/norm_l2-pca_None-min_cluster_size_20-min_samples_5-method_leaf/report.csv"
df = pd.read_csv(report_csv)

In [ ]:
df[df.cluster_label == 61].input_image_key.iloc[-1]

In [ ]:
# d7562801b9c502d17ff52e5ddb7687ba3241146faceaa101520a67c6b9e54f99
image_key = "d7562801b9c502d17ff52e5ddb7687ba3241146faceaa101520a67c6b9e54f99"
mask = (df.cluster_label == 61) & (df.input_image_key == image_key)
df[mask]

In [ ]:
keys, images = next(read_image_shard("imagenet-label-55-000000.tar", 128, {}))

our_idx = keys.index(image_key)
image = images[our_idx]

plt.imshow(np.array(image))
plt.show()

In [ ]:
make_overlay_heatmap(model, layer_name, [channel, 6, 2], image)

In [ ]:
batch = transform(image)[None]
acts = InputOutputModelSnapshot.get_activations(batch, model, ["mixed4d"])


In [ ]:
# set this to feature viz thing
# 6,2 is the position of interest
positions = [[6,2], [6,4], [6,6], [7,5], [8,6]]

expected_on_pos = [acts["mixed4d"]["output"][:,:,y,x]  for (y,x) in positions]
# all_pos.shape
# expected_mixed4d = acts["mixed4d"]["output"][:,:,6,2].clone()

In [ ]:
# we want to make an objective which has MSE on this for 6,2
expected_mixed4d.shape

In [ ]:
thresholds = list(range(0, 500, 25))
svizs = render.render_vis(model, patch_across_channel("mixed4d", expected_on_pos[:1], positions[:1]), verbose=True, show_image=False, thresholds=thresholds)
plt.imshow(svizs[-1][0])

In [ ]:
# not found thing

# expected_on_pos = [acts["mixed4d"]["output"][:,:,y,x]  for (y,x) in positions]
not_found_act=  [acts["mixed4d"]["output"][:, :, 6, 5]]
not_found_pos = [[6,5]]
thresholds = list(range(0, 500, 25))
svizs = render.render_vis(model, patch_across_channel("mixed4d", not_found_act, not_found_pos), verbose=True, show_image=False, thresholds=thresholds)

plt.imshow(svizs[-1][0])

In [ ]:
# not found thing

# expected_on_pos = [acts["mixed4d"]["output"][:,:,y,x]  for (y,x) in positions]
not_found_act=  [acts["mixed4d"]["output"][:, :, 10, 8]]
not_found_pos = [[10,8]]
thresholds = list(range(0, 500, 25))
svizs = render.render_vis(model, patch_across_channel("mixed4d", not_found_act, not_found_pos), verbose=True, show_image=False, thresholds=thresholds)

plt.imshow(svizs[-1][0])

In [ ]:
thresholds = list(range(0, 500, 25))
svizs = render.render_vis(model, patch_across_channel("mixed4d", expected_on_pos[1:2], positions[1:2]), verbose=True, show_image=False, thresholds=thresholds)

plt.imshow(svizs[-1][0])

In [ ]:
thresholds = list(range(0, 500, 25))
svizs = render.render_vis(model, patch_across_channel("mixed4d", expected_on_pos, positions), verbose=True, show_image=False, thresholds=thresholds)
plt.imshow(svizs[-2][0])

In [ ]:
svizs = render.render_vis(model, exact_tensor_all_chans("mixed4d", acts["mixed4d"]["output"]), verbose=True, show_image=False, thresholds=thresholds)
plt.imshow(svizs[-1][0])

## some other category

749 label has letters. we'll try that out now.  

In [ ]:
! ls

In [ ]:
! mv ~/Downloads/000000.tar imagenet-label-749-000000.tar

In [ ]:
# d7562801b9c502d17ff52e5ddb7687ba3241146faceaa101520a67c6b9e54f99
# mask = (df.cluster_label == 61) & (df.input_image_key == image_key)
df[(df.cluster_label == 24) & (df.imagenet_label == 749)].iloc[0].input_image_key

In [ ]:
image_key = "f486225b0dcd5a5d6efc7376670ffa174a05cae6a932301e58f9bb85af2fcc96"
mask = (df.cluster_label == 24) & (df.input_image_key == image_key)
df[mask]

In [ ]:
keys, images = next(read_image_shard("imagenet-label-749-000000.tar", 128, {}))

our_idx = keys.index(image_key)
image = images[our_idx]

plt.imshow(np.array(image))
plt.show()

In [ ]:
make_overlay_heatmap(model, layer_name, [channel, 4, 7], image)

In [ ]:
batch = transform(image)[None]
acts = InputOutputModelSnapshot.get_activations(batch, model, ["mixed4d"])


In [ ]:
positions = [[4,2], [4,7], [5,7], [4,10]]
expected_on_pos = [acts["mixed4d"]["output"][:,:,y,x]  for (y,x) in positions]

In [ ]:
thresholds = list(range(0, 500, 25))
svizs = render.render_vis(model, patch_across_channel("mixed4d", expected_on_pos, positions), verbose=True, show_image=False, thresholds=thresholds)

plt.imshow(svizs[-1][0])

A problem is that we would always have "something". we are taking a big chunk of the last output.  
Lets now capture the activation of our main layer and channel, and only keep the letters, we would like to see how much the output activation captures it.  

In [ ]:
batch = transform(image)[None]
acts = InputOutputModelSnapshot.get_activations(batch, model, [layer_name])
our_channel_acts = acts[layer_name]["output"][0][channel]
our_channel_acts.shape

In [ ]:
positions = [(4,2), (4,7), (5,7), (4,10)]
vals = [our_channel_acts[y,x][None] for (y,x) in positions]
# cant see a very useful pattern here but lets see
positions, vals

In [ ]:
vals[0].shape

In [ ]:
# we now only want to do the activations fo these 4 positions

thresholds = list(range(0, 500, 25))
svizs = render.render_vis(model, mse_at_multiple_positions(layer_name, channel, positions, vals), verbose=True, show_image=False, thresholds=thresholds)

plt.imshow(svizs[-1][0])
# mse_at_multiple_positions(layer, n_channel, positions, values, batch=None)            

### Diversity 


We would like to test if adding a diversity objective changes stuff. can we reliably reproduce this across examples?  
I would want a counter-example, a noise point might not give the same result (although, the points are consumed downstream by other neurons too, so it is possible that we might not find anything useful)    

Diversity objective does

In [ ]:

batch_param_f = lambda: param.image(128, batch=4)

thresholds = list(range(0, 500, 25))
obj = patch_across_channel("mixed4d", expected_on_pos, positions) - 1e2 * objectives.diversity(layer_name)kkkkkkkk
svizs = render.render_vis(model, obj, batch_param_f, verbose=True, show_image=False, thresholds=thresholds)


_, axes = plt.subplots(1, 4, figsize=(10,4))
for a, v in zip(axes, svizs[-1]):
    a.imshow(v)

plt.show()

Can we find a counter example? the 10th row is flat enough, does it have any useful features?  


Below, i feel it can be background, or noise. im not sure, not very conclusive.  

In [ ]:
positions = [(10,1), (10,3), (10,5), (10,7)]
vals = [our_channel_acts[y,x][None] for (y,x) in positions]
# cant see a very useful pattern here but lets see
positions, vals

In [ ]:
batch_param_f = lambda: param.image(128, batch=4)

thresholds = list(range(0, 500, 25))
obj = patch_across_channel("mixed4d", vals, positions) - 1e2 * objectives.diversity(layer_name)
svizs = render.render_vis(model, obj, batch_param_f, verbose=True, show_image=False, thresholds=thresholds)


_, axes = plt.subplots(1, 4, figsize=(10,4))
for a, v in zip(axes, svizs[-1]):
    a.imshow(v)

plt.show()

## human skin


Categories:

- 985 and 977

In [ ]:
! mv ~/Downloads/000000.tar ./imagenet-label-977-000000.tar

In [ ]:
report_csv = Path.home() / "Downloads/lucent-reports/overlay-reports-hyperparameter-grid-search/norm_l2-pca_None-min_cluster_size_20-min_samples_5-method_leaf/report.csv"
df = pd.read_csv(report_csv)

In [ ]:
# image_key = "f486225b0dcd5a5d6efc7376670ffa174a05cae6a932301e58f9bb85af2fcc96"
mask = (df.cluster_label == 41) & (df.imagenet_label == 977)
df[mask].iloc[3].input_image_key

In [ ]:
# d7562801b9c502d17ff52e5ddb7687ba3241146faceaa101520a67c6b9e54f99
image_key = "23961f30bdf9ea9e2d6c80058362be86a233191e699bad826d061a8b11c80d32"
mask = (df.cluster_label == 41) & (df.input_image_key == image_key)
df[mask]

In [ ]:
! mv ~/Downloads/000000.tar imagenet-label-985-000000.tar

In [ ]:
keys, images = next(read_image_shard("imagenet-label-977-000000.tar", 128, {}))

our_idx = keys.index(image_key)
image = images[our_idx]

plt.imshow(np.array(image))
plt.show()

In [ ]:
make_overlay_heatmap(model, layer_name, [channel, 10, 7], image)

In [ ]:
batch = transform(image)[None]
acts = InputOutputModelSnapshot.get_activations(batch, model, ["mixed4d"])


In [ ]:
positions = [[7,6], [8,6], [10,7]]
expected_on_pos = [acts["mixed4d"]["output"][:,:,y,x]  for (y,x) in positions]

In [ ]:
batch_param_f = lambda: param.image(128, batch=4)

thresholds = list(range(0, 500, 25))
obj = patch_across_channel("mixed4d", expected_on_pos, positions) - 1e2 * objectives.diversity(layer_name)
svizs = render.render_vis(model, obj, batch_param_f, verbose=True, show_image=False, thresholds=thresholds)


_, axes = plt.subplots(1, 4, figsize=(10,4))
for a, v in zip(axes, svizs[-1]):
    a.imshow(v)

plt.show()

In [ ]:
batch_param_f = lambda: param.image(128, batch=4)

thresholds = list(range(0, 500, 25))
obj = patch_across_channel("mixed4d", expected_on_pos[:1], positions[:1]) - 1e2 * objectives.diversity(layer_name)
svizs = render.render_vis(model, obj, batch_param_f, verbose=True, show_image=False, thresholds=thresholds)


_, axes = plt.subplots(1, 4, figsize=(10,4))
for a, v in zip(axes, svizs[-1]):
    a.imshow(v)

plt.show()

In [ ]:
len(df[mask])

In [ ]:
import random

random.randint(0, 19)

In [ ]:
import random
from uuid import uuid4

def generate_feature_viz_plots(cluster_label, imagenet_label, shard_path, name, df):
    thresholds = (64,)

    
    mask = (df.cluster_label == cluster_label) & (df.imagenet_label == imagenet_label)
    print("checking", cluster_label, imagenet_label)
    df.head()
    filtered_df = df[mask]
    
    row_idx = random.randint(0, len(filtered_df)-1)
    image_key = filtered_df[mask].iloc[row_idx].input_image_key
    keys, images = next(read_image_shard(shard_path, 128, {}))

    our_idx = keys.index(image_key)
    image = images[our_idx]
    batch = transform(image)[None]
    acts = InputOutputModelSnapshot.get_activations(batch, model, ["mixed4d"])
    positions = [
        (tup.y_position, tup.x_position)
        for tup in df[mask & (df.input_image_key == image_key)][["y_position", "x_position"]].itertuples()
    ]
    expected_on_pos = [acts["mixed4d"]["output"][:,:,y,x]  for (y,x) in positions]

    pos, vec = positions[0], expected_on_pos[0]

    
    batch_param_f = lambda: param.image(128, batch=4)
    
    obj = patch_across_channel("mixed4d", [vec], [pos]) - 1e2 * objectives.diversity(layer_name)
    svizs = render.render_vis(model, obj, batch_param_f, verbose=True, show_image=False, thresholds=thresholds)
    
    _, axes = plt.subplots(1, 5, figsize=(20,5))
    axes[0].imshow(np.array(make_overlay_heatmap(model, layer_name, [channel, pos[0], pos[1]], image)))
    v = svizs[-1]
    axes[1].imshow(v[0])
    axes[2].imshow(v[1])
    axes[3].imshow(v[2])
    axes[4].imshow(v[3])
    dest_dir = Path("feature-viz-auto") / f"{name}_{cluster_label}"
    dest_dir.mkdir(parents=True, exist_ok=True)
    
    plt.savefig(dest_dir / f"imagenet_label_{imagenet_label}_{uuid4()}.png", bbox_inches="tight")
    plt.close()
    # plt.show()

In [ ]:
tests = [
    {"cid": 41, "iid": [782, 977], "name": "bg"},
    {"cid": 58, "iid": [268, 295], "name": "bg"},
    {"cid": 53, "iid": [18, 105], "name": "dog-legs"},
    {"cid": 51, "iid": [391, 395], "name": "criss-cross"},
    {"cid": 24, "iid": [774, 706], "name": "letters"},
    {"cid": 59, "iid": [266, 265], "name": "cars"},
    {"cid": 45, "iid": [64, 27], "name": "animal-stomach"},
    {"cid": 31, "iid": [595, 817], "name": "human-face-top"},
    {"cid": 43, "iid": [754, 738], "name": "food"},
    {"cid": 50, "iid": [360, 364], "name": "mountain"},
    {"cid": 48, "iid": [833, 926], "name": "legs-cloth?"},
    {"cid": 60, "iid": [159, 118], "name": "snout"},
    {"cid": 61, "iid": [174, 10], "name": "cat"},
    {"cid": 40, "iid": [746, 913], "name": "mushroom"},
    {"cid": 32, "iid": [803, 354], "name": "human-faces-2"},
    {"cid": 23, "iid": [329, 389], "name": "corn-like-pattern"},
    {"cid": 29, "iid": [479, 491], "name": "snake"},
    {"cid": 44, "iid": [823, 844], "name": "curry"},
    {"cid": 33, "iid": [447, 449], "name": "fish"},
    {"cid": 23, "iid": [955, 504], "name": "thin-cylindrical"},
    {"cid": 18, "iid": [497, 494], "name": "maybe-croc-like"},
    {"cid": 52, "iid": [689, 426], "name": "water"},
    {"cid": 46, "iid": [72, 66], "name": "white-dog-stomach"},
    {"cid": 13, "iid": [551, 858], "name": "keyboard"},
    {"cid": 19, "iid": [524, 518], "name": "clock-edge"},
]

In [ ]:
report_csv = Path.home() / "Downloads/lucent-reports/overlay-reports-hyperparameter-grid-search/norm_l2-pca_None-min_cluster_size_20-min_samples_5-method_leaf/report.csv"
df = pd.read_csv(report_csv)

In [ ]:
for i, t in enumerate(tests):
    
    cluster_label, name = t["cid"], t["name"]
    for imagenet_label in t["iid"]:
        for j in range(2):
            # 2 samples per category
            shard_path = Path(f"./imagenet-label-{imagenet_label}-000000.tar")
            if not shard_path.exists():
                ! aws s3 cp s3://narang99-private/lucent-workdir/images/{imagenet_label}/000000.tar ./imagenet-label-{imagenet_label}-000000.tar
            print("############", name)
            generate_feature_viz_plots(cluster_label, imagenet_label, shard_path, name, df)

    if i > 2:
        break

In [ ]:
cluster_label = 41
imagenet_label = 985
shard_path = f"imagenet-label-{imagenet_label}-000000.tar"

generate_feature_viz_plots(cluster_label, imagenet_label, shard_path, df)

In [ ]:
cluster_label = 41
imagenet_label = 977
shard_path = f"imagenet-label-{imagenet_label}-000000.tar"

generate_feature_viz_plots(cluster_label, imagenet_label, shard_path, df)

In [ ]:
cluster_label = 41
imagenet_label = 977
shard_path = f"imagenet-label-{imagenet_label}-000000.tar"

generate_feature_viz_plots(cluster_label, imagenet_label, shard_path, df)

# Feature viz

In [ ]:

svizs = render.render_vis(model, objectives.channel("mixed4e_1x1_pre_relu_conv", 55), verbose=True, show_image=False)


timg = get_batch_from_feature_vis(svizs[0][0])

plt.imshow(svizs[0][0])

# the range seems to be present
# for max at least, its still hard for it, but not too bad
# so a more stricter objective is not winning at all
acts = InputOutputModelSnapshot.get_activations(timg, model, ["mixed4e_1x1_pre_relu_conv"])
acts["mixed4e_1x1_pre_relu_conv"]["output"][0, 55]

Now, with the neg channel objective, same exercise    
The numbers are actually much more negative than what we expected for human faces, its outside the human range.  
We do have some 60s, lets check them out?  

# inception data fixing


In [ ]:
from collections import OrderedDict


IMAGENET2012_CLASSES = OrderedDict(
    {
        "n01440764": "tench, Tinca tinca",
        "n01443537": "goldfish, Carassius auratus",
        "n01484850": "great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias",
        "n01491361": "tiger shark, Galeocerdo cuvieri",
        "n01494475": "hammerhead, hammerhead shark",
        "n01496331": "electric ray, crampfish, numbfish, torpedo",
        "n01498041": "stingray",
        "n01514668": "cock",
        "n01514859": "hen",
        "n01518878": "ostrich, Struthio camelus",
        "n01530575": "brambling, Fringilla montifringilla",
        "n01531178": "goldfinch, Carduelis carduelis",
        "n01532829": "house finch, linnet, Carpodacus mexicanus",
        "n01534433": "junco, snowbird",
        "n01537544": "indigo bunting, indigo finch, indigo bird, Passerina cyanea",
        "n01558993": "robin, American robin, Turdus migratorius",
        "n01560419": "bulbul",
        "n01580077": "jay",
        "n01582220": "magpie",
        "n01592084": "chickadee",
        "n01601694": "water ouzel, dipper",
        "n01608432": "kite",
        "n01614925": "bald eagle, American eagle, Haliaeetus leucocephalus",
        "n01616318": "vulture",
        "n01622779": "great grey owl, great gray owl, Strix nebulosa",
        "n01629819": "European fire salamander, Salamandra salamandra",
        "n01630670": "common newt, Triturus vulgaris",
        "n01631663": "eft",
        "n01632458": "spotted salamander, Ambystoma maculatum",
        "n01632777": "axolotl, mud puppy, Ambystoma mexicanum",
        "n01641577": "bullfrog, Rana catesbeiana",
        "n01644373": "tree frog, tree-frog",
        "n01644900": "tailed frog, bell toad, ribbed toad, tailed toad, Ascaphus trui",
        "n01664065": "loggerhead, loggerhead turtle, Caretta caretta",
        "n01665541": "leatherback turtle, leatherback, leathery turtle, Dermochelys coriacea",
        "n01667114": "mud turtle",
        "n01667778": "terrapin",
        "n01669191": "box turtle, box tortoise",
        "n01675722": "banded gecko",
        "n01677366": "common iguana, iguana, Iguana iguana",
        "n01682714": "American chameleon, anole, Anolis carolinensis",
        "n01685808": "whiptail, whiptail lizard",
        "n01687978": "agama",
        "n01688243": "frilled lizard, Chlamydosaurus kingi",
        "n01689811": "alligator lizard",
        "n01692333": "Gila monster, Heloderma suspectum",
        "n01693334": "green lizard, Lacerta viridis",
        "n01694178": "African chameleon, Chamaeleo chamaeleon",
        "n01695060": "Komodo dragon, Komodo lizard, dragon lizard, giant lizard, Varanus komodoensis",
        "n01697457": "African crocodile, Nile crocodile, Crocodylus niloticus",
        "n01698640": "American alligator, Alligator mississipiensis",
        "n01704323": "triceratops",
        "n01728572": "thunder snake, worm snake, Carphophis amoenus",
        "n01728920": "ringneck snake, ring-necked snake, ring snake",
        "n01729322": "hognose snake, puff adder, sand viper",
        "n01729977": "green snake, grass snake",
        "n01734418": "king snake, kingsnake",
        "n01735189": "garter snake, grass snake",
        "n01737021": "water snake",
        "n01739381": "vine snake",
        "n01740131": "night snake, Hypsiglena torquata",
        "n01742172": "boa constrictor, Constrictor constrictor",
        "n01744401": "rock python, rock snake, Python sebae",
        "n01748264": "Indian cobra, Naja naja",
        "n01749939": "green mamba",
        "n01751748": "sea snake",
        "n01753488": "horned viper, cerastes, sand viper, horned asp, Cerastes cornutus",
        "n01755581": "diamondback, diamondback rattlesnake, Crotalus adamanteus",
        "n01756291": "sidewinder, horned rattlesnake, Crotalus cerastes",
        "n01768244": "trilobite",
        "n01770081": "harvestman, daddy longlegs, Phalangium opilio",
        "n01770393": "scorpion",
        "n01773157": "black and gold garden spider, Argiope aurantia",
        "n01773549": "barn spider, Araneus cavaticus",
        "n01773797": "garden spider, Aranea diademata",
        "n01774384": "black widow, Latrodectus mactans",
        "n01774750": "tarantula",
        "n01775062": "wolf spider, hunting spider",
        "n01776313": "tick",
        "n01784675": "centipede",
        "n01795545": "black grouse",
        "n01796340": "ptarmigan",
        "n01797886": "ruffed grouse, partridge, Bonasa umbellus",
        "n01798484": "prairie chicken, prairie grouse, prairie fowl",
        "n01806143": "peacock",
        "n01806567": "quail",
        "n01807496": "partridge",
        "n01817953": "African grey, African gray, Psittacus erithacus",
        "n01818515": "macaw",
        "n01819313": "sulphur-crested cockatoo, Kakatoe galerita, Cacatua galerita",
        "n01820546": "lorikeet",
        "n01824575": "coucal",
        "n01828970": "bee eater",
        "n01829413": "hornbill",
        "n01833805": "hummingbird",
        "n01843065": "jacamar",
        "n01843383": "toucan",
        "n01847000": "drake",
        "n01855032": "red-breasted merganser, Mergus serrator",
        "n01855672": "goose",
        "n01860187": "black swan, Cygnus atratus",
        "n01871265": "tusker",
        "n01872401": "echidna, spiny anteater, anteater",
        "n01873310": "platypus, duckbill, duckbilled platypus, duck-billed platypus, Ornithorhynchus anatinus",
        "n01877812": "wallaby, brush kangaroo",
        "n01882714": "koala, koala bear, kangaroo bear, native bear, Phascolarctos cinereus",
        "n01883070": "wombat",
        "n01910747": "jellyfish",
        "n01914609": "sea anemone, anemone",
        "n01917289": "brain coral",
        "n01924916": "flatworm, platyhelminth",
        "n01930112": "nematode, nematode worm, roundworm",
        "n01943899": "conch",
        "n01944390": "snail",
        "n01945685": "slug",
        "n01950731": "sea slug, nudibranch",
        "n01955084": "chiton, coat-of-mail shell, sea cradle, polyplacophore",
        "n01968897": "chambered nautilus, pearly nautilus, nautilus",
        "n01978287": "Dungeness crab, Cancer magister",
        "n01978455": "rock crab, Cancer irroratus",
        "n01980166": "fiddler crab",
        "n01981276": "king crab, Alaska crab, Alaskan king crab, Alaska king crab, Paralithodes camtschatica",
        "n01983481": "American lobster, Northern lobster, Maine lobster, Homarus americanus",
        "n01984695": "spiny lobster, langouste, rock lobster, crawfish, crayfish, sea crawfish",
        "n01985128": "crayfish, crawfish, crawdad, crawdaddy",
        "n01986214": "hermit crab",
        "n01990800": "isopod",
        "n02002556": "white stork, Ciconia ciconia",
        "n02002724": "black stork, Ciconia nigra",
        "n02006656": "spoonbill",
        "n02007558": "flamingo",
        "n02009229": "little blue heron, Egretta caerulea",
        "n02009912": "American egret, great white heron, Egretta albus",
        "n02011460": "bittern",
        "n02012849": "crane",
        "n02013706": "limpkin, Aramus pictus",
        "n02017213": "European gallinule, Porphyrio porphyrio",
        "n02018207": "American coot, marsh hen, mud hen, water hen, Fulica americana",
        "n02018795": "bustard",
        "n02025239": "ruddy turnstone, Arenaria interpres",
        "n02027492": "red-backed sandpiper, dunlin, Erolia alpina",
        "n02028035": "redshank, Tringa totanus",
        "n02033041": "dowitcher",
        "n02037110": "oystercatcher, oyster catcher",
        "n02051845": "pelican",
        "n02056570": "king penguin, Aptenodytes patagonica",
        "n02058221": "albatross, mollymawk",
        "n02066245": "grey whale, gray whale, devilfish, Eschrichtius gibbosus, Eschrichtius robustus",
        "n02071294": "killer whale, killer, orca, grampus, sea wolf, Orcinus orca",
        "n02074367": "dugong, Dugong dugon",
        "n02077923": "sea lion",
        "n02085620": "Chihuahua",
        "n02085782": "Japanese spaniel",
        "n02085936": "Maltese dog, Maltese terrier, Maltese",
        "n02086079": "Pekinese, Pekingese, Peke",
        "n02086240": "Shih-Tzu",
        "n02086646": "Blenheim spaniel",
        "n02086910": "papillon",
        "n02087046": "toy terrier",
        "n02087394": "Rhodesian ridgeback",
        "n02088094": "Afghan hound, Afghan",
        "n02088238": "basset, basset hound",
        "n02088364": "beagle",
        "n02088466": "bloodhound, sleuthhound",
        "n02088632": "bluetick",
        "n02089078": "black-and-tan coonhound",
        "n02089867": "Walker hound, Walker foxhound",
        "n02089973": "English foxhound",
        "n02090379": "redbone",
        "n02090622": "borzoi, Russian wolfhound",
        "n02090721": "Irish wolfhound",
        "n02091032": "Italian greyhound",
        "n02091134": "whippet",
        "n02091244": "Ibizan hound, Ibizan Podenco",
        "n02091467": "Norwegian elkhound, elkhound",
        "n02091635": "otterhound, otter hound",
        "n02091831": "Saluki, gazelle hound",
        "n02092002": "Scottish deerhound, deerhound",
        "n02092339": "Weimaraner",
        "n02093256": "Staffordshire bullterrier, Staffordshire bull terrier",
        "n02093428": "American Staffordshire terrier, Staffordshire terrier, American pit bull terrier, pit bull terrier",
        "n02093647": "Bedlington terrier",
        "n02093754": "Border terrier",
        "n02093859": "Kerry blue terrier",
        "n02093991": "Irish terrier",
        "n02094114": "Norfolk terrier",
        "n02094258": "Norwich terrier",
        "n02094433": "Yorkshire terrier",
        "n02095314": "wire-haired fox terrier",
        "n02095570": "Lakeland terrier",
        "n02095889": "Sealyham terrier, Sealyham",
        "n02096051": "Airedale, Airedale terrier",
        "n02096177": "cairn, cairn terrier",
        "n02096294": "Australian terrier",
        "n02096437": "Dandie Dinmont, Dandie Dinmont terrier",
        "n02096585": "Boston bull, Boston terrier",
        "n02097047": "miniature schnauzer",
        "n02097130": "giant schnauzer",
        "n02097209": "standard schnauzer",
        "n02097298": "Scotch terrier, Scottish terrier, Scottie",
        "n02097474": "Tibetan terrier, chrysanthemum dog",
        "n02097658": "silky terrier, Sydney silky",
        "n02098105": "soft-coated wheaten terrier",
        "n02098286": "West Highland white terrier",
        "n02098413": "Lhasa, Lhasa apso",
        "n02099267": "flat-coated retriever",
        "n02099429": "curly-coated retriever",
        "n02099601": "golden retriever",
        "n02099712": "Labrador retriever",
        "n02099849": "Chesapeake Bay retriever",
        "n02100236": "German short-haired pointer",
        "n02100583": "vizsla, Hungarian pointer",
        "n02100735": "English setter",
        "n02100877": "Irish setter, red setter",
        "n02101006": "Gordon setter",
        "n02101388": "Brittany spaniel",
        "n02101556": "clumber, clumber spaniel",
        "n02102040": "English springer, English springer spaniel",
        "n02102177": "Welsh springer spaniel",
        "n02102318": "cocker spaniel, English cocker spaniel, cocker",
        "n02102480": "Sussex spaniel",
        "n02102973": "Irish water spaniel",
        "n02104029": "kuvasz",
        "n02104365": "schipperke",
        "n02105056": "groenendael",
        "n02105162": "malinois",
        "n02105251": "briard",
        "n02105412": "kelpie",
        "n02105505": "komondor",
        "n02105641": "Old English sheepdog, bobtail",
        "n02105855": "Shetland sheepdog, Shetland sheep dog, Shetland",
        "n02106030": "collie",
        "n02106166": "Border collie",
        "n02106382": "Bouvier des Flandres, Bouviers des Flandres",
        "n02106550": "Rottweiler",
        "n02106662": "German shepherd, German shepherd dog, German police dog, alsatian",
        "n02107142": "Doberman, Doberman pinscher",
        "n02107312": "miniature pinscher",
        "n02107574": "Greater Swiss Mountain dog",
        "n02107683": "Bernese mountain dog",
        "n02107908": "Appenzeller",
        "n02108000": "EntleBucher",
        "n02108089": "boxer",
        "n02108422": "bull mastiff",
        "n02108551": "Tibetan mastiff",
        "n02108915": "French bulldog",
        "n02109047": "Great Dane",
        "n02109525": "Saint Bernard, St Bernard",
        "n02109961": "Eskimo dog, husky",
        "n02110063": "malamute, malemute, Alaskan malamute",
        "n02110185": "Siberian husky",
        "n02110341": "dalmatian, coach dog, carriage dog",
        "n02110627": "affenpinscher, monkey pinscher, monkey dog",
        "n02110806": "basenji",
        "n02110958": "pug, pug-dog",
        "n02111129": "Leonberg",
        "n02111277": "Newfoundland, Newfoundland dog",
        "n02111500": "Great Pyrenees",
        "n02111889": "Samoyed, Samoyede",
        "n02112018": "Pomeranian",
        "n02112137": "chow, chow chow",
        "n02112350": "keeshond",
        "n02112706": "Brabancon griffon",
        "n02113023": "Pembroke, Pembroke Welsh corgi",
        "n02113186": "Cardigan, Cardigan Welsh corgi",
        "n02113624": "toy poodle",
        "n02113712": "miniature poodle",
        "n02113799": "standard poodle",
        "n02113978": "Mexican hairless",
        "n02114367": "timber wolf, grey wolf, gray wolf, Canis lupus",
        "n02114548": "white wolf, Arctic wolf, Canis lupus tundrarum",
        "n02114712": "red wolf, maned wolf, Canis rufus, Canis niger",
        "n02114855": "coyote, prairie wolf, brush wolf, Canis latrans",
        "n02115641": "dingo, warrigal, warragal, Canis dingo",
        "n02115913": "dhole, Cuon alpinus",
        "n02116738": "African hunting dog, hyena dog, Cape hunting dog, Lycaon pictus",
        "n02117135": "hyena, hyaena",
        "n02119022": "red fox, Vulpes vulpes",
        "n02119789": "kit fox, Vulpes macrotis",
        "n02120079": "Arctic fox, white fox, Alopex lagopus",
        "n02120505": "grey fox, gray fox, Urocyon cinereoargenteus",
        "n02123045": "tabby, tabby cat",
        "n02123159": "tiger cat",
        "n02123394": "Persian cat",
        "n02123597": "Siamese cat, Siamese",
        "n02124075": "Egyptian cat",
        "n02125311": "cougar, puma, catamount, mountain lion, painter, panther, Felis concolor",
        "n02127052": "lynx, catamount",
        "n02128385": "leopard, Panthera pardus",
        "n02128757": "snow leopard, ounce, Panthera uncia",
        "n02128925": "jaguar, panther, Panthera onca, Felis onca",
        "n02129165": "lion, king of beasts, Panthera leo",
        "n02129604": "tiger, Panthera tigris",
        "n02130308": "cheetah, chetah, Acinonyx jubatus",
        "n02132136": "brown bear, bruin, Ursus arctos",
        "n02133161": "American black bear, black bear, Ursus americanus, Euarctos americanus",
        "n02134084": "ice bear, polar bear, Ursus Maritimus, Thalarctos maritimus",
        "n02134418": "sloth bear, Melursus ursinus, Ursus ursinus",
        "n02137549": "mongoose",
        "n02138441": "meerkat, mierkat",
        "n02165105": "tiger beetle",
        "n02165456": "ladybug, ladybeetle, lady beetle, ladybird, ladybird beetle",
        "n02167151": "ground beetle, carabid beetle",
        "n02168699": "long-horned beetle, longicorn, longicorn beetle",
        "n02169497": "leaf beetle, chrysomelid",
        "n02172182": "dung beetle",
        "n02174001": "rhinoceros beetle",
        "n02177972": "weevil",
        "n02190166": "fly",
        "n02206856": "bee",
        "n02219486": "ant, emmet, pismire",
        "n02226429": "grasshopper, hopper",
        "n02229544": "cricket",
        "n02231487": "walking stick, walkingstick, stick insect",
        "n02233338": "cockroach, roach",
        "n02236044": "mantis, mantid",
        "n02256656": "cicada, cicala",
        "n02259212": "leafhopper",
        "n02264363": "lacewing, lacewing fly",
        "n02268443": "dragonfly, darning needle, devil's darning needle, sewing needle, snake feeder, snake doctor, mosquito hawk, skeeter hawk",
        "n02268853": "damselfly",
        "n02276258": "admiral",
        "n02277742": "ringlet, ringlet butterfly",
        "n02279972": "monarch, monarch butterfly, milkweed butterfly, Danaus plexippus",
        "n02280649": "cabbage butterfly",
        "n02281406": "sulphur butterfly, sulfur butterfly",
        "n02281787": "lycaenid, lycaenid butterfly",
        "n02317335": "starfish, sea star",
        "n02319095": "sea urchin",
        "n02321529": "sea cucumber, holothurian",
        "n02325366": "wood rabbit, cottontail, cottontail rabbit",
        "n02326432": "hare",
        "n02328150": "Angora, Angora rabbit",
        "n02342885": "hamster",
        "n02346627": "porcupine, hedgehog",
        "n02356798": "fox squirrel, eastern fox squirrel, Sciurus niger",
        "n02361337": "marmot",
        "n02363005": "beaver",
        "n02364673": "guinea pig, Cavia cobaya",
        "n02389026": "sorrel",
        "n02391049": "zebra",
        "n02395406": "hog, pig, grunter, squealer, Sus scrofa",
        "n02396427": "wild boar, boar, Sus scrofa",
        "n02397096": "warthog",
        "n02398521": "hippopotamus, hippo, river horse, Hippopotamus amphibius",
        "n02403003": "ox",
        "n02408429": "water buffalo, water ox, Asiatic buffalo, Bubalus bubalis",
        "n02410509": "bison",
        "n02412080": "ram, tup",
        "n02415577": "bighorn, bighorn sheep, cimarron, Rocky Mountain bighorn, Rocky Mountain sheep, Ovis canadensis",
        "n02417914": "ibex, Capra ibex",
        "n02422106": "hartebeest",
        "n02422699": "impala, Aepyceros melampus",
        "n02423022": "gazelle",
        "n02437312": "Arabian camel, dromedary, Camelus dromedarius",
        "n02437616": "llama",
        "n02441942": "weasel",
        "n02442845": "mink",
        "n02443114": "polecat, fitch, foulmart, foumart, Mustela putorius",
        "n02443484": "black-footed ferret, ferret, Mustela nigripes",
        "n02444819": "otter",
        "n02445715": "skunk, polecat, wood pussy",
        "n02447366": "badger",
        "n02454379": "armadillo",
        "n02457408": "three-toed sloth, ai, Bradypus tridactylus",
        "n02480495": "orangutan, orang, orangutang, Pongo pygmaeus",
        "n02480855": "gorilla, Gorilla gorilla",
        "n02481823": "chimpanzee, chimp, Pan troglodytes",
        "n02483362": "gibbon, Hylobates lar",
        "n02483708": "siamang, Hylobates syndactylus, Symphalangus syndactylus",
        "n02484975": "guenon, guenon monkey",
        "n02486261": "patas, hussar monkey, Erythrocebus patas",
        "n02486410": "baboon",
        "n02487347": "macaque",
        "n02488291": "langur",
        "n02488702": "colobus, colobus monkey",
        "n02489166": "proboscis monkey, Nasalis larvatus",
        "n02490219": "marmoset",
        "n02492035": "capuchin, ringtail, Cebus capucinus",
        "n02492660": "howler monkey, howler",
        "n02493509": "titi, titi monkey",
        "n02493793": "spider monkey, Ateles geoffroyi",
        "n02494079": "squirrel monkey, Saimiri sciureus",
        "n02497673": "Madagascar cat, ring-tailed lemur, Lemur catta",
        "n02500267": "indri, indris, Indri indri, Indri brevicaudatus",
        "n02504013": "Indian elephant, Elephas maximus",
        "n02504458": "African elephant, Loxodonta africana",
        "n02509815": "lesser panda, red panda, panda, bear cat, cat bear, Ailurus fulgens",
        "n02510455": "giant panda, panda, panda bear, coon bear, Ailuropoda melanoleuca",
        "n02514041": "barracouta, snoek",
        "n02526121": "eel",
        "n02536864": "coho, cohoe, coho salmon, blue jack, silver salmon, Oncorhynchus kisutch",
        "n02606052": "rock beauty, Holocanthus tricolor",
        "n02607072": "anemone fish",
        "n02640242": "sturgeon",
        "n02641379": "gar, garfish, garpike, billfish, Lepisosteus osseus",
        "n02643566": "lionfish",
        "n02655020": "puffer, pufferfish, blowfish, globefish",
        "n02666196": "abacus",
        "n02667093": "abaya",
        "n02669723": "academic gown, academic robe, judge's robe",
        "n02672831": "accordion, piano accordion, squeeze box",
        "n02676566": "acoustic guitar",
        "n02687172": "aircraft carrier, carrier, flattop, attack aircraft carrier",
        "n02690373": "airliner",
        "n02692877": "airship, dirigible",
        "n02699494": "altar",
        "n02701002": "ambulance",
        "n02704792": "amphibian, amphibious vehicle",
        "n02708093": "analog clock",
        "n02727426": "apiary, bee house",
        "n02730930": "apron",
        "n02747177": "ashcan, trash can, garbage can, wastebin, ash bin, ash-bin, ashbin, dustbin, trash barrel, trash bin",
        "n02749479": "assault rifle, assault gun",
        "n02769748": "backpack, back pack, knapsack, packsack, rucksack, haversack",
        "n02776631": "bakery, bakeshop, bakehouse",
        "n02777292": "balance beam, beam",
        "n02782093": "balloon",
        "n02783161": "ballpoint, ballpoint pen, ballpen, Biro",
        "n02786058": "Band Aid",
        "n02787622": "banjo",
        "n02788148": "bannister, banister, balustrade, balusters, handrail",
        "n02790996": "barbell",
        "n02791124": "barber chair",
        "n02791270": "barbershop",
        "n02793495": "barn",
        "n02794156": "barometer",
        "n02795169": "barrel, cask",
        "n02797295": "barrow, garden cart, lawn cart, wheelbarrow",
        "n02799071": "baseball",
        "n02802426": "basketball",
        "n02804414": "bassinet",
        "n02804610": "bassoon",
        "n02807133": "bathing cap, swimming cap",
        "n02808304": "bath towel",
        "n02808440": "bathtub, bathing tub, bath, tub",
        "n02814533": "beach wagon, station wagon, wagon, estate car, beach waggon, station waggon, waggon",
        "n02814860": "beacon, lighthouse, beacon light, pharos",
        "n02815834": "beaker",
        "n02817516": "bearskin, busby, shako",
        "n02823428": "beer bottle",
        "n02823750": "beer glass",
        "n02825657": "bell cote, bell cot",
        "n02834397": "bib",
        "n02835271": "bicycle-built-for-two, tandem bicycle, tandem",
        "n02837789": "bikini, two-piece",
        "n02840245": "binder, ring-binder",
        "n02841315": "binoculars, field glasses, opera glasses",
        "n02843684": "birdhouse",
        "n02859443": "boathouse",
        "n02860847": "bobsled, bobsleigh, bob",
        "n02865351": "bolo tie, bolo, bola tie, bola",
        "n02869837": "bonnet, poke bonnet",
        "n02870880": "bookcase",
        "n02871525": "bookshop, bookstore, bookstall",
        "n02877765": "bottlecap",
        "n02879718": "bow",
        "n02883205": "bow tie, bow-tie, bowtie",
        "n02892201": "brass, memorial tablet, plaque",
        "n02892767": "brassiere, bra, bandeau",
        "n02894605": "breakwater, groin, groyne, mole, bulwark, seawall, jetty",
        "n02895154": "breastplate, aegis, egis",
        "n02906734": "broom",
        "n02909870": "bucket, pail",
        "n02910353": "buckle",
        "n02916936": "bulletproof vest",
        "n02917067": "bullet train, bullet",
        "n02927161": "butcher shop, meat market",
        "n02930766": "cab, hack, taxi, taxicab",
        "n02939185": "caldron, cauldron",
        "n02948072": "candle, taper, wax light",
        "n02950826": "cannon",
        "n02951358": "canoe",
        "n02951585": "can opener, tin opener",
        "n02963159": "cardigan",
        "n02965783": "car mirror",
        "n02966193": "carousel, carrousel, merry-go-round, roundabout, whirligig",
        "n02966687": "carpenter's kit, tool kit",
        "n02971356": "carton",
        "n02974003": "car wheel",
        "n02977058": "cash machine, cash dispenser, automated teller machine, automatic teller machine, automated teller, automatic teller, ATM",
        "n02978881": "cassette",
        "n02979186": "cassette player",
        "n02980441": "castle",
        "n02981792": "catamaran",
        "n02988304": "CD player",
        "n02992211": "cello, violoncello",
        "n02992529": "cellular telephone, cellular phone, cellphone, cell, mobile phone",
        "n02999410": "chain",
        "n03000134": "chainlink fence",
        "n03000247": "chain mail, ring mail, mail, chain armor, chain armour, ring armor, ring armour",
        "n03000684": "chain saw, chainsaw",
        "n03014705": "chest",
        "n03016953": "chiffonier, commode",
        "n03017168": "chime, bell, gong",
        "n03018349": "china cabinet, china closet",
        "n03026506": "Christmas stocking",
        "n03028079": "church, church building",
        "n03032252": "cinema, movie theater, movie theatre, movie house, picture palace",
        "n03041632": "cleaver, meat cleaver, chopper",
        "n03042490": "cliff dwelling",
        "n03045698": "cloak",
        "n03047690": "clog, geta, patten, sabot",
        "n03062245": "cocktail shaker",
        "n03063599": "coffee mug",
        "n03063689": "coffeepot",
        "n03065424": "coil, spiral, volute, whorl, helix",
        "n03075370": "combination lock",
        "n03085013": "computer keyboard, keypad",
        "n03089624": "confectionery, confectionary, candy store",
        "n03095699": "container ship, containership, container vessel",
        "n03100240": "convertible",
        "n03109150": "corkscrew, bottle screw",
        "n03110669": "cornet, horn, trumpet, trump",
        "n03124043": "cowboy boot",
        "n03124170": "cowboy hat, ten-gallon hat",
        "n03125729": "cradle",
        "n03126707": "crane2",
        "n03127747": "crash helmet",
        "n03127925": "crate",
        "n03131574": "crib, cot",
        "n03133878": "Crock Pot",
        "n03134739": "croquet ball",
        "n03141823": "crutch",
        "n03146219": "cuirass",
        "n03160309": "dam, dike, dyke",
        "n03179701": "desk",
        "n03180011": "desktop computer",
        "n03187595": "dial telephone, dial phone",
        "n03188531": "diaper, nappy, napkin",
        "n03196217": "digital clock",
        "n03197337": "digital watch",
        "n03201208": "dining table, board",
        "n03207743": "dishrag, dishcloth",
        "n03207941": "dishwasher, dish washer, dishwashing machine",
        "n03208938": "disk brake, disc brake",
        "n03216828": "dock, dockage, docking facility",
        "n03218198": "dogsled, dog sled, dog sleigh",
        "n03220513": "dome",
        "n03223299": "doormat, welcome mat",
        "n03240683": "drilling platform, offshore rig",
        "n03249569": "drum, membranophone, tympan",
        "n03250847": "drumstick",
        "n03255030": "dumbbell",
        "n03259280": "Dutch oven",
        "n03271574": "electric fan, blower",
        "n03272010": "electric guitar",
        "n03272562": "electric locomotive",
        "n03290653": "entertainment center",
        "n03291819": "envelope",
        "n03297495": "espresso maker",
        "n03314780": "face powder",
        "n03325584": "feather boa, boa",
        "n03337140": "file, file cabinet, filing cabinet",
        "n03344393": "fireboat",
        "n03345487": "fire engine, fire truck",
        "n03347037": "fire screen, fireguard",
        "n03355925": "flagpole, flagstaff",
        "n03372029": "flute, transverse flute",
        "n03376595": "folding chair",
        "n03379051": "football helmet",
        "n03384352": "forklift",
        "n03388043": "fountain",
        "n03388183": "fountain pen",
        "n03388549": "four-poster",
        "n03393912": "freight car",
        "n03394916": "French horn, horn",
        "n03400231": "frying pan, frypan, skillet",
        "n03404251": "fur coat",
        "n03417042": "garbage truck, dustcart",
        "n03424325": "gasmask, respirator, gas helmet",
        "n03425413": "gas pump, gasoline pump, petrol pump, island dispenser",
        "n03443371": "goblet",
        "n03444034": "go-kart",
        "n03445777": "golf ball",
        "n03445924": "golfcart, golf cart",
        "n03447447": "gondola",
        "n03447721": "gong, tam-tam",
        "n03450230": "gown",
        "n03452741": "grand piano, grand",
        "n03457902": "greenhouse, nursery, glasshouse",
        "n03459775": "grille, radiator grille",
        "n03461385": "grocery store, grocery, food market, market",
        "n03467068": "guillotine",
        "n03476684": "hair slide",
        "n03476991": "hair spray",
        "n03478589": "half track",
        "n03481172": "hammer",
        "n03482405": "hamper",
        "n03483316": "hand blower, blow dryer, blow drier, hair dryer, hair drier",
        "n03485407": "hand-held computer, hand-held microcomputer",
        "n03485794": "handkerchief, hankie, hanky, hankey",
        "n03492542": "hard disc, hard disk, fixed disk",
        "n03494278": "harmonica, mouth organ, harp, mouth harp",
        "n03495258": "harp",
        "n03496892": "harvester, reaper",
        "n03498962": "hatchet",
        "n03527444": "holster",
        "n03529860": "home theater, home theatre",
        "n03530642": "honeycomb",
        "n03532672": "hook, claw",
        "n03534580": "hoopskirt, crinoline",
        "n03535780": "horizontal bar, high bar",
        "n03538406": "horse cart, horse-cart",
        "n03544143": "hourglass",
        "n03584254": "iPod",
        "n03584829": "iron, smoothing iron",
        "n03590841": "jack-o'-lantern",
        "n03594734": "jean, blue jean, denim",
        "n03594945": "jeep, landrover",
        "n03595614": "jersey, T-shirt, tee shirt",
        "n03598930": "jigsaw puzzle",
        "n03599486": "jinrikisha, ricksha, rickshaw",
        "n03602883": "joystick",
        "n03617480": "kimono",
        "n03623198": "knee pad",
        "n03627232": "knot",
        "n03630383": "lab coat, laboratory coat",
        "n03633091": "ladle",
        "n03637318": "lampshade, lamp shade",
        "n03642806": "laptop, laptop computer",
        "n03649909": "lawn mower, mower",
        "n03657121": "lens cap, lens cover",
        "n03658185": "letter opener, paper knife, paperknife",
        "n03661043": "library",
        "n03662601": "lifeboat",
        "n03666591": "lighter, light, igniter, ignitor",
        "n03670208": "limousine, limo",
        "n03673027": "liner, ocean liner",
        "n03676483": "lipstick, lip rouge",
        "n03680355": "Loafer",
        "n03690938": "lotion",
        "n03691459": "loudspeaker, speaker, speaker unit, loudspeaker system, speaker system",
        "n03692522": "loupe, jeweler's loupe",
        "n03697007": "lumbermill, sawmill",
        "n03706229": "magnetic compass",
        "n03709823": "mailbag, postbag",
        "n03710193": "mailbox, letter box",
        "n03710637": "maillot",
        "n03710721": "maillot, tank suit",
        "n03717622": "manhole cover",
        "n03720891": "maraca",
        "n03721384": "marimba, xylophone",
        "n03724870": "mask",
        "n03729826": "matchstick",
        "n03733131": "maypole",
        "n03733281": "maze, labyrinth",
        "n03733805": "measuring cup",
        "n03742115": "medicine chest, medicine cabinet",
        "n03743016": "megalith, megalithic structure",
        "n03759954": "microphone, mike",
        "n03761084": "microwave, microwave oven",
        "n03763968": "military uniform",
        "n03764736": "milk can",
        "n03769881": "minibus",
        "n03770439": "miniskirt, mini",
        "n03770679": "minivan",
        "n03773504": "missile",
        "n03775071": "mitten",
        "n03775546": "mixing bowl",
        "n03776460": "mobile home, manufactured home",
        "n03777568": "Model T",
        "n03777754": "modem",
        "n03781244": "monastery",
        "n03782006": "monitor",
        "n03785016": "moped",
        "n03786901": "mortar",
        "n03787032": "mortarboard",
        "n03788195": "mosque",
        "n03788365": "mosquito net",
        "n03791053": "motor scooter, scooter",
        "n03792782": "mountain bike, all-terrain bike, off-roader",
        "n03792972": "mountain tent",
        "n03793489": "mouse, computer mouse",
        "n03794056": "mousetrap",
        "n03796401": "moving van",
        "n03803284": "muzzle",
        "n03804744": "nail",
        "n03814639": "neck brace",
        "n03814906": "necklace",
        "n03825788": "nipple",
        "n03832673": "notebook, notebook computer",
        "n03837869": "obelisk",
        "n03838899": "oboe, hautboy, hautbois",
        "n03840681": "ocarina, sweet potato",
        "n03841143": "odometer, hodometer, mileometer, milometer",
        "n03843555": "oil filter",
        "n03854065": "organ, pipe organ",
        "n03857828": "oscilloscope, scope, cathode-ray oscilloscope, CRO",
        "n03866082": "overskirt",
        "n03868242": "oxcart",
        "n03868863": "oxygen mask",
        "n03871628": "packet",
        "n03873416": "paddle, boat paddle",
        "n03874293": "paddlewheel, paddle wheel",
        "n03874599": "padlock",
        "n03876231": "paintbrush",
        "n03877472": "pajama, pyjama, pj's, jammies",
        "n03877845": "palace",
        "n03884397": "panpipe, pandean pipe, syrinx",
        "n03887697": "paper towel",
        "n03888257": "parachute, chute",
        "n03888605": "parallel bars, bars",
        "n03891251": "park bench",
        "n03891332": "parking meter",
        "n03895866": "passenger car, coach, carriage",
        "n03899768": "patio, terrace",
        "n03902125": "pay-phone, pay-station",
        "n03903868": "pedestal, plinth, footstall",
        "n03908618": "pencil box, pencil case",
        "n03908714": "pencil sharpener",
        "n03916031": "perfume, essence",
        "n03920288": "Petri dish",
        "n03924679": "photocopier",
        "n03929660": "pick, plectrum, plectron",
        "n03929855": "pickelhaube",
        "n03930313": "picket fence, paling",
        "n03930630": "pickup, pickup truck",
        "n03933933": "pier",
        "n03935335": "piggy bank, penny bank",
        "n03937543": "pill bottle",
        "n03938244": "pillow",
        "n03942813": "ping-pong ball",
        "n03944341": "pinwheel",
        "n03947888": "pirate, pirate ship",
        "n03950228": "pitcher, ewer",
        "n03954731": "plane, carpenter's plane, woodworking plane",
        "n03956157": "planetarium",
        "n03958227": "plastic bag",
        "n03961711": "plate rack",
        "n03967562": "plow, plough",
        "n03970156": "plunger, plumber's helper",
        "n03976467": "Polaroid camera, Polaroid Land camera",
        "n03976657": "pole",
        "n03977966": "police van, police wagon, paddy wagon, patrol wagon, wagon, black Maria",
        "n03980874": "poncho",
        "n03982430": "pool table, billiard table, snooker table",
        "n03983396": "pop bottle, soda bottle",
        "n03991062": "pot, flowerpot",
        "n03992509": "potter's wheel",
        "n03995372": "power drill",
        "n03998194": "prayer rug, prayer mat",
        "n04004767": "printer",
        "n04005630": "prison, prison house",
        "n04008634": "projectile, missile",
        "n04009552": "projector",
        "n04019541": "puck, hockey puck",
        "n04023962": "punching bag, punch bag, punching ball, punchball",
        "n04026417": "purse",
        "n04033901": "quill, quill pen",
        "n04033995": "quilt, comforter, comfort, puff",
        "n04037443": "racer, race car, racing car",
        "n04039381": "racket, racquet",
        "n04040759": "radiator",
        "n04041544": "radio, wireless",
        "n04044716": "radio telescope, radio reflector",
        "n04049303": "rain barrel",
        "n04065272": "recreational vehicle, RV, R.V.",
        "n04067472": "reel",
        "n04069434": "reflex camera",
        "n04070727": "refrigerator, icebox",
        "n04074963": "remote control, remote",
        "n04081281": "restaurant, eating house, eating place, eatery",
        "n04086273": "revolver, six-gun, six-shooter",
        "n04090263": "rifle",
        "n04099969": "rocking chair, rocker",
        "n04111531": "rotisserie",
        "n04116512": "rubber eraser, rubber, pencil eraser",
        "n04118538": "rugby ball",
        "n04118776": "rule, ruler",
        "n04120489": "running shoe",
        "n04125021": "safe",
        "n04127249": "safety pin",
        "n04131690": "saltshaker, salt shaker",
        "n04133789": "sandal",
        "n04136333": "sarong",
        "n04141076": "sax, saxophone",
        "n04141327": "scabbard",
        "n04141975": "scale, weighing machine",
        "n04146614": "school bus",
        "n04147183": "schooner",
        "n04149813": "scoreboard",
        "n04152593": "screen, CRT screen",
        "n04153751": "screw",
        "n04154565": "screwdriver",
        "n04162706": "seat belt, seatbelt",
        "n04179913": "sewing machine",
        "n04192698": "shield, buckler",
        "n04200800": "shoe shop, shoe-shop, shoe store",
        "n04201297": "shoji",
        "n04204238": "shopping basket",
        "n04204347": "shopping cart",
        "n04208210": "shovel",
        "n04209133": "shower cap",
        "n04209239": "shower curtain",
        "n04228054": "ski",
        "n04229816": "ski mask",
        "n04235860": "sleeping bag",
        "n04238763": "slide rule, slipstick",
        "n04239074": "sliding door",
        "n04243546": "slot, one-armed bandit",
        "n04251144": "snorkel",
        "n04252077": "snowmobile",
        "n04252225": "snowplow, snowplough",
        "n04254120": "soap dispenser",
        "n04254680": "soccer ball",
        "n04254777": "sock",
        "n04258138": "solar dish, solar collector, solar furnace",
        "n04259630": "sombrero",
        "n04263257": "soup bowl",
        "n04264628": "space bar",
        "n04265275": "space heater",
        "n04266014": "space shuttle",
        "n04270147": "spatula",
        "n04273569": "speedboat",
        "n04275548": "spider web, spider's web",
        "n04277352": "spindle",
        "n04285008": "sports car, sport car",
        "n04286575": "spotlight, spot",
        "n04296562": "stage",
        "n04310018": "steam locomotive",
        "n04311004": "steel arch bridge",
        "n04311174": "steel drum",
        "n04317175": "stethoscope",
        "n04325704": "stole",
        "n04326547": "stone wall",
        "n04328186": "stopwatch, stop watch",
        "n04330267": "stove",
        "n04332243": "strainer",
        "n04335435": "streetcar, tram, tramcar, trolley, trolley car",
        "n04336792": "stretcher",
        "n04344873": "studio couch, day bed",
        "n04346328": "stupa, tope",
        "n04347754": "submarine, pigboat, sub, U-boat",
        "n04350905": "suit, suit of clothes",
        "n04355338": "sundial",
        "n04355933": "sunglass",
        "n04356056": "sunglasses, dark glasses, shades",
        "n04357314": "sunscreen, sunblock, sun blocker",
        "n04366367": "suspension bridge",
        "n04367480": "swab, swob, mop",
        "n04370456": "sweatshirt",
        "n04371430": "swimming trunks, bathing trunks",
        "n04371774": "swing",
        "n04372370": "switch, electric switch, electrical switch",
        "n04376876": "syringe",
        "n04380533": "table lamp",
        "n04389033": "tank, army tank, armored combat vehicle, armoured combat vehicle",
        "n04392985": "tape player",
        "n04398044": "teapot",
        "n04399382": "teddy, teddy bear",
        "n04404412": "television, television system",
        "n04409515": "tennis ball",
        "n04417672": "thatch, thatched roof",
        "n04418357": "theater curtain, theatre curtain",
        "n04423845": "thimble",
        "n04428191": "thresher, thrasher, threshing machine",
        "n04429376": "throne",
        "n04435653": "tile roof",
        "n04442312": "toaster",
        "n04443257": "tobacco shop, tobacconist shop, tobacconist",
        "n04447861": "toilet seat",
        "n04456115": "torch",
        "n04458633": "totem pole",
        "n04461696": "tow truck, tow car, wrecker",
        "n04462240": "toyshop",
        "n04465501": "tractor",
        "n04467665": "trailer truck, tractor trailer, trucking rig, rig, articulated lorry, semi",
        "n04476259": "tray",
        "n04479046": "trench coat",
        "n04482393": "tricycle, trike, velocipede",
        "n04483307": "trimaran",
        "n04485082": "tripod",
        "n04486054": "triumphal arch",
        "n04487081": "trolleybus, trolley coach, trackless trolley",
        "n04487394": "trombone",
        "n04493381": "tub, vat",
        "n04501370": "turnstile",
        "n04505470": "typewriter keyboard",
        "n04507155": "umbrella",
        "n04509417": "unicycle, monocycle",
        "n04515003": "upright, upright piano",
        "n04517823": "vacuum, vacuum cleaner",
        "n04522168": "vase",
        "n04523525": "vault",
        "n04525038": "velvet",
        "n04525305": "vending machine",
        "n04532106": "vestment",
        "n04532670": "viaduct",
        "n04536866": "violin, fiddle",
        "n04540053": "volleyball",
        "n04542943": "waffle iron",
        "n04548280": "wall clock",
        "n04548362": "wallet, billfold, notecase, pocketbook",
        "n04550184": "wardrobe, closet, press",
        "n04552348": "warplane, military plane",
        "n04553703": "washbasin, handbasin, washbowl, lavabo, wash-hand basin",
        "n04554684": "washer, automatic washer, washing machine",
        "n04557648": "water bottle",
        "n04560804": "water jug",
        "n04562935": "water tower",
        "n04579145": "whiskey jug",
        "n04579432": "whistle",
        "n04584207": "wig",
        "n04589890": "window screen",
        "n04590129": "window shade",
        "n04591157": "Windsor tie",
        "n04591713": "wine bottle",
        "n04592741": "wing",
        "n04596742": "wok",
        "n04597913": "wooden spoon",
        "n04599235": "wool, woolen, woollen",
        "n04604644": "worm fence, snake fence, snake-rail fence, Virginia fence",
        "n04606251": "wreck",
        "n04612504": "yawl",
        "n04613696": "yurt",
        "n06359193": "web site, website, internet site, site",
        "n06596364": "comic book",
        "n06785654": "crossword puzzle, crossword",
        "n06794110": "street sign",
        "n06874185": "traffic light, traffic signal, stoplight",
        "n07248320": "book jacket, dust cover, dust jacket, dust wrapper",
        "n07565083": "menu",
        "n07579787": "plate",
        "n07583066": "guacamole",
        "n07584110": "consomme",
        "n07590611": "hot pot, hotpot",
        "n07613480": "trifle",
        "n07614500": "ice cream, icecream",
        "n07615774": "ice lolly, lolly, lollipop, popsicle",
        "n07684084": "French loaf",
        "n07693725": "bagel, beigel",
        "n07695742": "pretzel",
        "n07697313": "cheeseburger",
        "n07697537": "hotdog, hot dog, red hot",
        "n07711569": "mashed potato",
        "n07714571": "head cabbage",
        "n07714990": "broccoli",
        "n07715103": "cauliflower",
        "n07716358": "zucchini, courgette",
        "n07716906": "spaghetti squash",
        "n07717410": "acorn squash",
        "n07717556": "butternut squash",
        "n07718472": "cucumber, cuke",
        "n07718747": "artichoke, globe artichoke",
        "n07720875": "bell pepper",
        "n07730033": "cardoon",
        "n07734744": "mushroom",
        "n07742313": "Granny Smith",
        "n07745940": "strawberry",
        "n07747607": "orange",
        "n07749582": "lemon",
        "n07753113": "fig",
        "n07753275": "pineapple, ananas",
        "n07753592": "banana",
        "n07754684": "jackfruit, jak, jack",
        "n07760859": "custard apple",
        "n07768694": "pomegranate",
        "n07802026": "hay",
        "n07831146": "carbonara",
        "n07836838": "chocolate sauce, chocolate syrup",
        "n07860988": "dough",
        "n07871810": "meat loaf, meatloaf",
        "n07873807": "pizza, pizza pie",
        "n07875152": "potpie",
        "n07880968": "burrito",
        "n07892512": "red wine",
        "n07920052": "espresso",
        "n07930864": "cup",
        "n07932039": "eggnog",
        "n09193705": "alp",
        "n09229709": "bubble",
        "n09246464": "cliff, drop, drop-off",
        "n09256479": "coral reef",
        "n09288635": "geyser",
        "n09332890": "lakeside, lakeshore",
        "n09399592": "promontory, headland, head, foreland",
        "n09421951": "sandbar, sand bar",
        "n09428293": "seashore, coast, seacoast, sea-coast",
        "n09468604": "valley, vale",
        "n09472597": "volcano",
        "n09835506": "ballplayer, baseball player",
        "n10148035": "groom, bridegroom",
        "n10565667": "scuba diver",
        "n11879895": "rapeseed",
        "n11939491": "daisy",
        "n12057211": "yellow lady's slipper, yellow lady-slipper, Cypripedium calceolus, Cypripedium parviflorum",
        "n12144580": "corn",
        "n12267677": "acorn",
        "n12620546": "hip, rose hip, rosehip",
        "n12768682": "buckeye, horse chestnut, conker",
        "n12985857": "coral fungus",
        "n12998815": "agaric",
        "n13037406": "gyromitra",
        "n13040303": "stinkhorn, carrion fungus",
        "n13044778": "earthstar",
        "n13052670": "hen-of-the-woods, hen of the woods, Polyporus frondosus, Grifola frondosa",
        "n13054560": "bolete",
        "n13133613": "ear, spike, capitulum",
        "n15075141": "toilet tissue, toilet paper, bathroom tissue",
    }
)

In [ ]:
with open("/Users/hariomnarang/Desktop/personal/lucent/lucent/modelzoo/misc/old_imagenet_labels.txt") as f:
    old_labels_content = f.readlines()

In [ ]:
old_imagenet_class_id_by_label = {}
for line in old_labels_content:
    comps = line.split(" ")
    old_imagenet_class_id, old_inceptionv1_label = comps[0].strip(), int(comps[1].strip())
    old_imagenet_class_id_by_label[old_imagenet_class_id] = old_inceptionv1_label

In [ ]:
old_imagenet_class_id_by_label

In [ ]:
new_imagenet_class_id_by_label = {}
for i, (new_imagenet_class_id, desc) in enumerate(IMAGENET2012_CLASSES.items()):
    new_imagenet_class_id_by_label[new_imagenet_class_id] = i

In [ ]:
new_imagenet_class_id_by_label

In [ ]:
old_label_to_new_label = {}
not_there_count = 0
for new_class_id, new_label in new_imagenet_class_id_by_label.items():
    if new_class_id in old_imagenet_class_id_by_label:        
        old_label = old_imagenet_class_id_by_label[new_class_id]
        old_label_to_new_label[old_label] = new_label
        # not_there_count += 1

In [ ]:
old_label_to_new_label

In [ ]:
old_label_to_new_label

In [ ]:
import json
with open("./old_label_to_new_label.json", "w") as f:
    json.dump(old_label_to_new_label, f, indent=2)

# feature viz try

In [ ]:
from torch import optim
import torch.optim.lr_scheduler as lr_scheduler


image_f, params, transform_f, objective_f, hook, features = get_feature_viz_input(
    model, 
    # mse_at_position("mixed4e_1x1_pre_relu_conv", 55, (7,7), torch.tensor([-400.])), 
    patch_across_channel("mixed4d", expected_mixed4d, (6,2))
    # exact_tensor("mixed4e_1x1_pre_relu_conv", 55, torch.ones((1,14,14))*(-400), batch=None)
)

thresholds = list(range(0, 512, 20))
num_epochs = max(thresholds)
optimizer = optim.Adam(params, lr=5e-2)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

images = []

model(transform_f(image_f()))
print("Initial loss: {:.3f}".format(objective_f(hook)))

for i in range(1, num_epochs + 1):
    optimizer.zero_grad()
    try:
        model(transform_f(image_f()))
    except RuntimeError as ex:
        if i == 1:
            # Only display the warning message
            # on the first iteration, no need to do that
            # every iteration
            warnings.warn(
                "Some layers could not be computed because the size of the "
                "image is not big enough. It is fine, as long as the non"
                "computed layers are not used in the objective function"
                f"(exception details: '{ex}')"
            )
    loss = objective_f(hook)
    loss.backward()

    

    optimizer.step()
    scheduler.step()

    if i in thresholds:
        image = tensor_to_img_array(image_f())            
        print("Loss at step {}: {:.3f}".format(i, objective_f(hook)))
        images.append(image)

# Clear hooks
for module_hook in features.values():
    del module_hook.module._forward_hooks[module_hook.hook.id]


In [ ]:
plt.imshow(images[-1][0])

In [ ]:

# svizs = render.render_vis(model, neg_channel("mixed4e_1x1_pre_relu_conv", 55), verbose=True, show_image=False)


timg = get_batch_from_feature_vis(svizs[5][0])


# the range seems to be present
# for max at least, its still hard for it, but not too bad
# so a more stricter objective is not winning at all
acts = InputOutputModelSnapshot.get_activations(timg, model, ["mixed4e_1x1_pre_relu_conv"])
print(acts["mixed4e_1x1_pre_relu_conv"]["output"][0, 55, 7 ,7])

plt.imshow(images[-1][0])
plt.show()


In [ ]:

value = torch.ones((1,14,14))*200
svizs = render.render_vis(model, exact_tensor("mixed4e_1x1_pre_relu_conv", 55, value), thresholds=list(range(0, 512, 20)), verbose=True, show_image=False)


timg = get_batch_from_feature_vis(svizs[0][0])

plt.imshow(svizs[0][0])

# the range seems to be present
# for max at least, its still hard for it, but not too bad
# so a more stricter objective is not winning at all
acts = InputOutputModelSnapshot.get_activations(timg, model, ["mixed4e_1x1_pre_relu_conv"])
acts["mixed4e_1x1_pre_relu_conv"]["output"][0, 55]

In [ ]:
from torch import optim


optimizer = lambda params: optim.Adam(params, lr=1e-3)

value = torch.ones((1,14,14))*200
svizs = render.render_vis(
    model, 
    mse_at_position("mixed4e_1x1_pre_relu_conv", 55, (7,7), torch.tensor([300.])), 
    thresholds=list(range(0, 2500, 100)), 
    verbose=True, 
    show_image=False,
    optimizer=optimizer,
)




In [ ]:
plt.imshow(svizs[22][0])

In [ ]:
timg = get_batch_from_feature_vis(svizs[23][0])

plt.imshow(svizs[0][0])

# the range seems to be present
# for max at least, its still hard for it, but not too bad
# so a more stricter objective is not winning at all
acts = InputOutputModelSnapshot.get_activations(timg, model, ["mixed4e_1x1_pre_relu_conv"])
acts["mixed4e_1x1_pre_relu_conv"]["output"][0, 55, 7, 7]

# csv preps

In [ ]:
import pandas as pd
p = Path.home() / "Desktop/personal/blog/pages/posts/2026-06-19-disentangling-mixed4e-55/reports/report_hdbscan-layer_mixed4e_1x1_pre_relu_conv-channel_55-min_cluser_size_20-alg_leaf/report.csv"
df = pd.read_csv(p)



print(pd.DataFrame({
    'Total samples': df['cluster_label'].value_counts(),
    'Number of unique images': df.groupby('cluster_label')['input_image_key'].nunique()
}).sort_values('Total samples', ascending=False))

In [ ]:
df.groupby("cluster_label")['input_image_key'].nunique()

In [ ]:
df.cluster_label.value_counts()

In [ ]:
df

In [ ]:
df.cluster_label.nunique()

In [ ]:
df[(df.imagenet_label == 281)].input_image_key.nunique()

In [ ]:
df[(df.imagenet_label == 281) & (df.cluster_label != -1)]

In [ ]:
pca_df = pd.read_csv(Path.home() / "Downloads" / "pca-report" / "report.csv")


In [ ]:
im_label_by_name = {
"0": "tench, Tinca tinca",
"1": "goldfish, Carassius auratus",
"2": "great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias",
"3": "tiger shark, Galeocerdo cuvieri",
"4": "hammerhead, hammerhead shark",
"5": "electric ray, crampfish, numbfish, torpedo",
"6": "stingray",
"7": "cock",
"8": "hen",
"9": "ostrich, Struthio camelus",
"10": "brambling, Fringilla montifringilla",
"11": "goldfinch, Carduelis carduelis",
"12": "house finch, linnet, Carpodacus mexicanus",
"13": "junco, snowbird",
"14": "indigo bunting, indigo finch, indigo bird, Passerina cyanea",
"15": "robin, American robin, Turdus migratorius",
"16": "bulbul",
"17": "jay",
"18": "magpie",
"19": "chickadee",
"20": "water ouzel, dipper",
"21": "kite",
"22": "bald eagle, American eagle, Haliaeetus leucocephalus",
"23": "vulture",
"24": "great grey owl, great gray owl, Strix nebulosa",
"25": "European fire salamander, Salamandra salamandra",
"26": "common newt, Triturus vulgaris",
"27": "eft",
"28": "spotted salamander, Ambystoma maculatum",
"29": "axolotl, mud puppy, Ambystoma mexicanum",
"30": "bullfrog, Rana catesbeiana",
"31": "tree frog, tree-frog",
"32": "tailed frog, bell toad, ribbed toad, tailed toad, Ascaphus trui",
"33": "loggerhead, loggerhead turtle, Caretta caretta",
"34": "leatherback turtle, leatherback, leathery turtle, Dermochelys coriacea",
"35": "mud turtle",
"36": "terrapin",
"37": "box turtle, box tortoise",
"38": "banded gecko",
"39": "common iguana, iguana, Iguana iguana",
"40": "American chameleon, anole, Anolis carolinensis",
"41": "whiptail, whiptail lizard",
"42": "agama",
"43": "frilled lizard, Chlamydosaurus kingi",
"44": "alligator lizard",
"45": "Gila monster, Heloderma suspectum",
"46": "green lizard, Lacerta viridis",
"47": "African chameleon, Chamaeleo chamaeleon",
"48": "Komodo dragon, Komodo lizard, dragon lizard, giant lizard, Varanus komodoensis",
"49": "African crocodile, Nile crocodile, Crocodylus niloticus",
"50": "American alligator, Alligator mississipiensis",
"51": "triceratops",
"52": "thunder snake, worm snake, Carphophis amoenus",
"53": "ringneck snake, ring-necked snake, ring snake",
"54": "hognose snake, puff adder, sand viper",
"55": "green snake, grass snake",
"56": "king snake, kingsnake",
"57": "garter snake, grass snake",
"58": "water snake",
"59": "vine snake",
"60": "night snake, Hypsiglena torquata",
"61": "boa constrictor, Constrictor constrictor",
"62": "rock python, rock snake, Python sebae",
"63": "Indian cobra, Naja naja",
"64": "green mamba",
"65": "sea snake",
"66": "horned viper, cerastes, sand viper, horned asp, Cerastes cornutus",
"67": "diamondback, diamondback rattlesnake, Crotalus adamanteus",
"68": "sidewinder, horned rattlesnake, Crotalus cerastes",
"69": "trilobite",
"70": "harvestman, daddy longlegs, Phalangium opilio",
"71": "scorpion",
"72": "black and gold garden spider, Argiope aurantia",
"73": "barn spider, Araneus cavaticus",
"74": "garden spider, Aranea diademata",
"75": "black widow, Latrodectus mactans",
"76": "tarantula",
"77": "wolf spider, hunting spider",
"78": "tick",
"79": "centipede",
"80": "black grouse",
"81": "ptarmigan",
"82": "ruffed grouse, partridge, Bonasa umbellus",
"83": "prairie chicken, prairie grouse, prairie fowl",
"84": "peacock",
"85": "quail",
"86": "partridge",
"87": "African grey, African gray, Psittacus erithacus",
"88": "macaw",
"89": "sulphur-crested cockatoo, Kakatoe galerita, Cacatua galerita",
"90": "lorikeet",
"91": "coucal",
"92": "bee eater",
"93": "hornbill",
"94": "hummingbird",
"95": "jacamar",
"96": "toucan",
"97": "drake",
"98": "red-breasted merganser, Mergus serrator",
"99": "goose",
"100": "black swan, Cygnus atratus",
"101": "tusker",
"102": "echidna, spiny anteater, anteater",
"103": "platypus, duckbill, duckbilled platypus, duck-billed platypus, Ornithorhynchus anatinus",
"104": "wallaby, brush kangaroo",
"105": "koala, koala bear, kangaroo bear, native bear, Phascolarctos cinereus",
"106": "wombat",
"107": "jellyfish",
"108": "sea anemone, anemone",
"109": "brain coral",
"110": "flatworm, platyhelminth",
"111": "nematode, nematode worm, roundworm",
"112": "conch",
"113": "snail",
"114": "slug",
"115": "sea slug, nudibranch",
"116": "chiton, coat-of-mail shell, sea cradle, polyplacophore",
"117": "chambered nautilus, pearly nautilus, nautilus",
"118": "Dungeness crab, Cancer magister",
"119": "rock crab, Cancer irroratus",
"120": "fiddler crab",
"121": "king crab, Alaska crab, Alaskan king crab, Alaska king crab, Paralithodes camtschatica",
"122": "American lobster, Northern lobster, Maine lobster, Homarus americanus",
"123": "spiny lobster, langouste, rock lobster, crawfish, crayfish, sea crawfish",
"124": "crayfish, crawfish, crawdad, crawdaddy",
"125": "hermit crab",
"126": "isopod",
"127": "white stork, Ciconia ciconia",
"128": "black stork, Ciconia nigra",
"129": "spoonbill",
"130": "flamingo",
"131": "little blue heron, Egretta caerulea",
"132": "American egret, great white heron, Egretta albus",
"133": "bittern",
"134": "crane",
"135": "limpkin, Aramus pictus",
"136": "European gallinule, Porphyrio porphyrio",
"137": "American coot, marsh hen, mud hen, water hen, Fulica americana",
"138": "bustard",
"139": "ruddy turnstone, Arenaria interpres",
"140": "red-backed sandpiper, dunlin, Erolia alpina",
"141": "redshank, Tringa totanus",
"142": "dowitcher",
"143": "oystercatcher, oyster catcher",
"144": "pelican",
"145": "king penguin, Aptenodytes patagonica",
"146": "albatross, mollymawk",
"147": "grey whale, gray whale, devilfish, Eschrichtius gibbosus, Eschrichtius robustus",
"148": "killer whale, killer, orca, grampus, sea wolf, Orcinus orca",
"149": "dugong, Dugong dugon",
"150": "sea lion",
"151": "Chihuahua",
"152": "Japanese spaniel",
"153": "Maltese dog, Maltese terrier, Maltese",
"154": "Pekinese, Pekingese, Peke",
"155": "Shih-Tzu",
"156": "Blenheim spaniel",
"157": "papillon",
"158": "toy terrier",
"159": "Rhodesian ridgeback",
"160": "Afghan hound, Afghan",
"161": "basset, basset hound",
"162": "beagle",
"163": "bloodhound, sleuthhound",
"164": "bluetick",
"165": "black-and-tan coonhound",
"166": "Walker hound, Walker foxhound",
"167": "English foxhound",
"168": "redbone",
"169": "borzoi, Russian wolfhound",
"170": "Irish wolfhound",
"171": "Italian greyhound",
"172": "whippet",
"173": "Ibizan hound, Ibizan Podenco",
"174": "Norwegian elkhound, elkhound",
"175": "otterhound, otter hound",
"176": "Saluki, gazelle hound",
"177": "Scottish deerhound, deerhound",
"178": "Weimaraner",
"179": "Staffordshire bullterrier, Staffordshire bull terrier",
"180": "American Staffordshire terrier, Staffordshire terrier, American pit bull terrier, pit bull terrier",
"181": "Bedlington terrier",
"182": "Border terrier",
"183": "Kerry blue terrier",
"184": "Irish terrier",
"185": "Norfolk terrier",
"186": "Norwich terrier",
"187": "Yorkshire terrier",
"188": "wire-haired fox terrier",
"189": "Lakeland terrier",
"190": "Sealyham terrier, Sealyham",
"191": "Airedale, Airedale terrier",
"192": "cairn, cairn terrier",
"193": "Australian terrier",
"194": "Dandie Dinmont, Dandie Dinmont terrier",
"195": "Boston bull, Boston terrier",
"196": "miniature schnauzer",
"197": "giant schnauzer",
"198": "standard schnauzer",
"199": "Scotch terrier, Scottish terrier, Scottie",
"200": "Tibetan terrier, chrysanthemum dog",
"201": "silky terrier, Sydney silky",
"202": "soft-coated wheaten terrier",
"203": "West Highland white terrier",
"204": "Lhasa, Lhasa apso",
"205": "flat-coated retriever",
"206": "curly-coated retriever",
"207": "golden retriever",
"208": "Labrador retriever",
"209": "Chesapeake Bay retriever",
"210": "German short-haired pointer",
"211": "vizsla, Hungarian pointer",
"212": "English setter",
"213": "Irish setter, red setter",
"214": "Gordon setter",
"215": "Brittany spaniel",
"216": "clumber, clumber spaniel",
"217": "English springer, English springer spaniel",
"218": "Welsh springer spaniel",
"219": "cocker spaniel, English cocker spaniel, cocker",
"220": "Sussex spaniel",
"221": "Irish water spaniel",
"222": "kuvasz",
"223": "schipperke",
"224": "groenendael",
"225": "malinois",
"226": "briard",
"227": "kelpie",
"228": "komondor",
"229": "Old English sheepdog, bobtail",
"230": "Shetland sheepdog, Shetland sheep dog, Shetland",
"231": "collie",
"232": "Border collie",
"233": "Bouvier des Flandres, Bouviers des Flandres",
"234": "Rottweiler",
"235": "German shepherd, German shepherd dog, German police dog, alsatian",
"236": "Doberman, Doberman pinscher",
"237": "miniature pinscher",
"238": "Greater Swiss Mountain dog",
"239": "Bernese mountain dog",
"240": "Appenzeller",
"241": "EntleBucher",
"242": "boxer",
"243": "bull mastiff",
"244": "Tibetan mastiff",
"245": "French bulldog",
"246": "Great Dane",
"247": "Saint Bernard, St Bernard",
"248": "Eskimo dog, husky",
"249": "malamute, malemute, Alaskan malamute",
"250": "Siberian husky",
"251": "dalmatian, coach dog, carriage dog",
"252": "affenpinscher, monkey pinscher, monkey dog",
"253": "basenji",
"254": "pug, pug-dog",
"255": "Leonberg",
"256": "Newfoundland, Newfoundland dog",
"257": "Great Pyrenees",
"258": "Samoyed, Samoyede",
"259": "Pomeranian",
"260": "chow, chow chow",
"261": "keeshond",
"262": "Brabancon griffon",
"263": "Pembroke, Pembroke Welsh corgi",
"264": "Cardigan, Cardigan Welsh corgi",
"265": "toy poodle",
"266": "miniature poodle",
"267": "standard poodle",
"268": "Mexican hairless",
"269": "timber wolf, grey wolf, gray wolf, Canis lupus",
"270": "white wolf, Arctic wolf, Canis lupus tundrarum",
"271": "red wolf, maned wolf, Canis rufus, Canis niger",
"272": "coyote, prairie wolf, brush wolf, Canis latrans",
"273": "dingo, warrigal, warragal, Canis dingo",
"274": "dhole, Cuon alpinus",
"275": "African hunting dog, hyena dog, Cape hunting dog, Lycaon pictus",
"276": "hyena, hyaena",
"277": "red fox, Vulpes vulpes",
"278": "kit fox, Vulpes macrotis",
"279": "Arctic fox, white fox, Alopex lagopus",
"280": "grey fox, gray fox, Urocyon cinereoargenteus",
"281": "tabby, tabby cat",
"282": "tiger cat",
"283": "Persian cat",
"284": "Siamese cat, Siamese",
"285": "Egyptian cat",
"286": "cougar, puma, catamount, mountain lion, painter, panther, Felis concolor",
"287": "lynx, catamount",
"288": "leopard, Panthera pardus",
"289": "snow leopard, ounce, Panthera uncia",
"290": "jaguar, panther, Panthera onca, Felis onca",
"291": "lion, king of beasts, Panthera leo",
"292": "tiger, Panthera tigris",
"293": "cheetah, chetah, Acinonyx jubatus",
"294": "brown bear, bruin, Ursus arctos",
"295": "American black bear, black bear, Ursus americanus, Euarctos americanus",
"296": "ice bear, polar bear, Ursus Maritimus, Thalarctos maritimus",
"297": "sloth bear, Melursus ursinus, Ursus ursinus",
"298": "mongoose",
"299": "meerkat, mierkat",
"300": "tiger beetle",
"301": "ladybug, ladybeetle, lady beetle, ladybird, ladybird beetle",
"302": "ground beetle, carabid beetle",
"303": "long-horned beetle, longicorn, longicorn beetle",
"304": "leaf beetle, chrysomelid",
"305": "dung beetle",
"306": "rhinoceros beetle",
"307": "weevil",
"308": "fly",
"309": "bee",
"310": "ant, emmet, pismire",
"311": "grasshopper, hopper",
"312": "cricket",
"313": "walking stick, walkingstick, stick insect",
"314": "cockroach, roach",
"315": "mantis, mantid",
"316": "cicada, cicala",
"317": "leafhopper",
"318": "lacewing, lacewing fly",
"319": "dragonfly, darning needle, devil's darning needle, sewing needle, snake feeder, snake doctor, mosquito hawk, skeeter hawk",
"320": "damselfly",
"321": "admiral",
"322": "ringlet, ringlet butterfly",
"323": "monarch, monarch butterfly, milkweed butterfly, Danaus plexippus",
"324": "cabbage butterfly",
"325": "sulphur butterfly, sulfur butterfly",
"326": "lycaenid, lycaenid butterfly",
"327": "starfish, sea star",
"328": "sea urchin",
"329": "sea cucumber, holothurian",
"330": "wood rabbit, cottontail, cottontail rabbit",
"331": "hare",
"332": "Angora, Angora rabbit",
"333": "hamster",
"334": "porcupine, hedgehog",
"335": "fox squirrel, eastern fox squirrel, Sciurus niger",
"336": "marmot",
"337": "beaver",
"338": "guinea pig, Cavia cobaya",
"339": "sorrel",
"340": "zebra",
"341": "hog, pig, grunter, squealer, Sus scrofa",
"342": "wild boar, boar, Sus scrofa",
"343": "warthog",
"344": "hippopotamus, hippo, river horse, Hippopotamus amphibius",
"345": "ox",
"346": "water buffalo, water ox, Asiatic buffalo, Bubalus bubalis",
"347": "bison",
"348": "ram, tup",
"349": "bighorn, bighorn sheep, cimarron, Rocky Mountain bighorn, Rocky Mountain sheep, Ovis canadensis",
"350": "ibex, Capra ibex",
"351": "hartebeest",
"352": "impala, Aepyceros melampus",
"353": "gazelle",
"354": "Arabian camel, dromedary, Camelus dromedarius",
"355": "llama",
"356": "weasel",
"357": "mink",
"358": "polecat, fitch, foulmart, foumart, Mustela putorius",
"359": "black-footed ferret, ferret, Mustela nigripes",
"360": "otter",
"361": "skunk, polecat, wood pussy",
"362": "badger",
"363": "armadillo",
"364": "three-toed sloth, ai, Bradypus tridactylus",
"365": "orangutan, orang, orangutang, Pongo pygmaeus",
"366": "gorilla, Gorilla gorilla",
"367": "chimpanzee, chimp, Pan troglodytes",
"368": "gibbon, Hylobates lar",
"369": "siamang, Hylobates syndactylus, Symphalangus syndactylus",
"370": "guenon, guenon monkey",
"371": "patas, hussar monkey, Erythrocebus patas",
"372": "baboon",
"373": "macaque",
"374": "langur",
"375": "colobus, colobus monkey",
"376": "proboscis monkey, Nasalis larvatus",
"377": "marmoset",
"378": "capuchin, ringtail, Cebus capucinus",
"379": "howler monkey, howler",
"380": "titi, titi monkey",
"381": "spider monkey, Ateles geoffroyi",
"382": "squirrel monkey, Saimiri sciureus",
"383": "Madagascar cat, ring-tailed lemur, Lemur catta",
"384": "indri, indris, Indri indri, Indri brevicaudatus",
"385": "Indian elephant, Elephas maximus",
"386": "African elephant, Loxodonta africana",
"387": "lesser panda, red panda, panda, bear cat, cat bear, Ailurus fulgens",
"388": "giant panda, panda, panda bear, coon bear, Ailuropoda melanoleuca",
"389": "barracouta, snoek",
"390": "eel",
"391": "coho, cohoe, coho salmon, blue jack, silver salmon, Oncorhynchus kisutch",
"392": "rock beauty, Holocanthus tricolor",
"393": "anemone fish",
"394": "sturgeon",
"395": "gar, garfish, garpike, billfish, Lepisosteus osseus",
"396": "lionfish",
"397": "puffer, pufferfish, blowfish, globefish",
"398": "abacus",
"399": "abaya",
"400": "academic gown, academic robe, judge's robe",
"401": "accordion, piano accordion, squeeze box",
"402": "acoustic guitar",
"403": "aircraft carrier, carrier, flattop, attack aircraft carrier",
"404": "airliner",
"405": "airship, dirigible",
"406": "altar",
"407": "ambulance",
"408": "amphibian, amphibious vehicle",
"409": "analog clock",
"410": "apiary, bee house",
"411": "apron",
"412": "ashcan, trash can, garbage can, wastebin, ash bin, ash-bin, ashbin, dustbin, trash barrel, trash bin",
"413": "assault rifle, assault gun",
"414": "backpack, back pack, knapsack, packsack, rucksack, haversack",
"415": "bakery, bakeshop, bakehouse",
"416": "balance beam, beam",
"417": "balloon",
"418": "ballpoint, ballpoint pen, ballpen, Biro",
"419": "Band Aid",
"420": "banjo",
"421": "bannister, banister, balustrade, balusters, handrail",
"422": "barbell",
"423": "barber chair",
"424": "barbershop",
"425": "barn",
"426": "barometer",
"427": "barrel, cask",
"428": "barrow, garden cart, lawn cart, wheelbarrow",
"429": "baseball",
"430": "basketball",
"431": "bassinet",
"432": "bassoon",
"433": "bathing cap, swimming cap",
"434": "bath towel",
"435": "bathtub, bathing tub, bath, tub",
"436": "beach wagon, station wagon, wagon, estate car, beach waggon, station waggon, waggon",
"437": "beacon, lighthouse, beacon light, pharos",
"438": "beaker",
"439": "bearskin, busby, shako",
"440": "beer bottle",
"441": "beer glass",
"442": "bell cote, bell cot",
"443": "bib",
"444": "bicycle-built-for-two, tandem bicycle, tandem",
"445": "bikini, two-piece",
"446": "binder, ring-binder",
"447": "binoculars, field glasses, opera glasses",
"448": "birdhouse",
"449": "boathouse",
"450": "bobsled, bobsleigh, bob",
"451": "bolo tie, bolo, bola tie, bola",
"452": "bonnet, poke bonnet",
"453": "bookcase",
"454": "bookshop, bookstore, bookstall",
"455": "bottlecap",
"456": "bow",
"457": "bow tie, bow-tie, bowtie",
"458": "brass, memorial tablet, plaque",
"459": "brassiere, bra, bandeau",
"460": "breakwater, groin, groyne, mole, bulwark, seawall, jetty",
"461": "breastplate, aegis, egis",
"462": "broom",
"463": "bucket, pail",
"464": "buckle",
"465": "bulletproof vest",
"466": "bullet train, bullet",
"467": "butcher shop, meat market",
"468": "cab, hack, taxi, taxicab",
"469": "caldron, cauldron",
"470": "candle, taper, wax light",
"471": "cannon",
"472": "canoe",
"473": "can opener, tin opener",
"474": "cardigan",
"475": "car mirror",
"476": "carousel, carrousel, merry-go-round, roundabout, whirligig",
"477": "carpenter's kit, tool kit",
"478": "carton",
"479": "car wheel",
"480": "cash machine, cash dispenser, automated teller machine, automatic teller machine, automated teller, automatic teller, ATM",
"481": "cassette",
"482": "cassette player",
"483": "castle",
"484": "catamaran",
"485": "CD player",
"486": "cello, violoncello",
"487": "cellular telephone, cellular phone, cellphone, cell, mobile phone",
"488": "chain",
"489": "chainlink fence",
"490": "chain mail, ring mail, mail, chain armor, chain armour, ring armor, ring armour",
"491": "chain saw, chainsaw",
"492": "chest",
"493": "chiffonier, commode",
"494": "chime, bell, gong",
"495": "china cabinet, china closet",
"496": "Christmas stocking",
"497": "church, church building",
"498": "cinema, movie theater, movie theatre, movie house, picture palace",
"499": "cleaver, meat cleaver, chopper",
"500": "cliff dwelling",
"501": "cloak",
"502": "clog, geta, patten, sabot",
"503": "cocktail shaker",
"504": "coffee mug",
"505": "coffeepot",
"506": "coil, spiral, volute, whorl, helix",
"507": "combination lock",
"508": "computer keyboard, keypad",
"509": "confectionery, confectionary, candy store",
"510": "container ship, containership, container vessel",
"511": "convertible",
"512": "corkscrew, bottle screw",
"513": "cornet, horn, trumpet, trump",
"514": "cowboy boot",
"515": "cowboy hat, ten-gallon hat",
"516": "cradle",
"517": "crane",
"518": "crash helmet",
"519": "crate",
"520": "crib, cot",
"521": "Crock Pot",
"522": "croquet ball",
"523": "crutch",
"524": "cuirass",
"525": "dam, dike, dyke",
"526": "desk",
"527": "desktop computer",
"528": "dial telephone, dial phone",
"529": "diaper, nappy, napkin",
"530": "digital clock",
"531": "digital watch",
"532": "dining table, board",
"533": "dishrag, dishcloth",
"534": "dishwasher, dish washer, dishwashing machine",
"535": "disk brake, disc brake",
"536": "dock, dockage, docking facility",
"537": "dogsled, dog sled, dog sleigh",
"538": "dome",
"539": "doormat, welcome mat",
"540": "drilling platform, offshore rig",
"541": "drum, membranophone, tympan",
"542": "drumstick",
"543": "dumbbell",
"544": "Dutch oven",
"545": "electric fan, blower",
"546": "electric guitar",
"547": "electric locomotive",
"548": "entertainment center",
"549": "envelope",
"550": "espresso maker",
"551": "face powder",
"552": "feather boa, boa",
"553": "file, file cabinet, filing cabinet",
"554": "fireboat",
"555": "fire engine, fire truck",
"556": "fire screen, fireguard",
"557": "flagpole, flagstaff",
"558": "flute, transverse flute",
"559": "folding chair",
"560": "football helmet",
"561": "forklift",
"562": "fountain",
"563": "fountain pen",
"564": "four-poster",
"565": "freight car",
"566": "French horn, horn",
"567": "frying pan, frypan, skillet",
"568": "fur coat",
"569": "garbage truck, dustcart",
"570": "gasmask, respirator, gas helmet",
"571": "gas pump, gasoline pump, petrol pump, island dispenser",
"572": "goblet",
"573": "go-kart",
"574": "golf ball",
"575": "golfcart, golf cart",
"576": "gondola",
"577": "gong, tam-tam",
"578": "gown",
"579": "grand piano, grand",
"580": "greenhouse, nursery, glasshouse",
"581": "grille, radiator grille",
"582": "grocery store, grocery, food market, market",
"583": "guillotine",
"584": "hair slide",
"585": "hair spray",
"586": "half track",
"587": "hammer",
"588": "hamper",
"589": "hand blower, blow dryer, blow drier, hair dryer, hair drier",
"590": "hand-held computer, hand-held microcomputer",
"591": "handkerchief, hankie, hanky, hankey",
"592": "hard disc, hard disk, fixed disk",
"593": "harmonica, mouth organ, harp, mouth harp",
"594": "harp",
"595": "harvester, reaper",
"596": "hatchet",
"597": "holster",
"598": "home theater, home theatre",
"599": "honeycomb",
"600": "hook, claw",
"601": "hoopskirt, crinoline",
"602": "horizontal bar, high bar",
"603": "horse cart, horse-cart",
"604": "hourglass",
"605": "iPod",
"606": "iron, smoothing iron",
"607": "jack-o'-lantern",
"608": "jean, blue jean, denim",
"609": "jeep, landrover",
"610": "jersey, T-shirt, tee shirt",
"611": "jigsaw puzzle",
"612": "jinrikisha, ricksha, rickshaw",
"613": "joystick",
"614": "kimono",
"615": "knee pad",
"616": "knot",
"617": "lab coat, laboratory coat",
"618": "ladle",
"619": "lampshade, lamp shade",
"620": "laptop, laptop computer",
"621": "lawn mower, mower",
"622": "lens cap, lens cover",
"623": "letter opener, paper knife, paperknife",
"624": "library",
"625": "lifeboat",
"626": "lighter, light, igniter, ignitor",
"627": "limousine, limo",
"628": "liner, ocean liner",
"629": "lipstick, lip rouge",
"630": "Loafer",
"631": "lotion",
"632": "loudspeaker, speaker, speaker unit, loudspeaker system, speaker system",
"633": "loupe, jeweler's loupe",
"634": "lumbermill, sawmill",
"635": "magnetic compass",
"636": "mailbag, postbag",
"637": "mailbox, letter box",
"638": "maillot",
"639": "maillot, tank suit",
"640": "manhole cover",
"641": "maraca",
"642": "marimba, xylophone",
"643": "mask",
"644": "matchstick",
"645": "maypole",
"646": "maze, labyrinth",
"647": "measuring cup",
"648": "medicine chest, medicine cabinet",
"649": "megalith, megalithic structure",
"650": "microphone, mike",
"651": "microwave, microwave oven",
"652": "military uniform",
"653": "milk can",
"654": "minibus",
"655": "miniskirt, mini",
"656": "minivan",
"657": "missile",
"658": "mitten",
"659": "mixing bowl",
"660": "mobile home, manufactured home",
"661": "Model T",
"662": "modem",
"663": "monastery",
"664": "monitor",
"665": "moped",
"666": "mortar",
"667": "mortarboard",
"668": "mosque",
"669": "mosquito net",
"670": "motor scooter, scooter",
"671": "mountain bike, all-terrain bike, off-roader",
"672": "mountain tent",
"673": "mouse, computer mouse",
"674": "mousetrap",
"675": "moving van",
"676": "muzzle",
"677": "nail",
"678": "neck brace",
"679": "necklace",
"680": "nipple",
"681": "notebook, notebook computer",
"682": "obelisk",
"683": "oboe, hautboy, hautbois",
"684": "ocarina, sweet potato",
"685": "odometer, hodometer, mileometer, milometer",
"686": "oil filter",
"687": "organ, pipe organ",
"688": "oscilloscope, scope, cathode-ray oscilloscope, CRO",
"689": "overskirt",
"690": "oxcart",
"691": "oxygen mask",
"692": "packet",
"693": "paddle, boat paddle",
"694": "paddlewheel, paddle wheel",
"695": "padlock",
"696": "paintbrush",
"697": "pajama, pyjama, pj's, jammies",
"698": "palace",
"699": "panpipe, pandean pipe, syrinx",
"700": "paper towel",
"701": "parachute, chute",
"702": "parallel bars, bars",
"703": "park bench",
"704": "parking meter",
"705": "passenger car, coach, carriage",
"706": "patio, terrace",
"707": "pay-phone, pay-station",
"708": "pedestal, plinth, footstall",
"709": "pencil box, pencil case",
"710": "pencil sharpener",
"711": "perfume, essence",
"712": "Petri dish",
"713": "photocopier",
"714": "pick, plectrum, plectron",
"715": "pickelhaube",
"716": "picket fence, paling",
"717": "pickup, pickup truck",
"718": "pier",
"719": "piggy bank, penny bank",
"720": "pill bottle",
"721": "pillow",
"722": "ping-pong ball",
"723": "pinwheel",
"724": "pirate, pirate ship",
"725": "pitcher, ewer",
"726": "plane, carpenter's plane, woodworking plane",
"727": "planetarium",
"728": "plastic bag",
"729": "plate rack",
"730": "plow, plough",
"731": "plunger, plumber's helper",
"732": "Polaroid camera, Polaroid Land camera",
"733": "pole",
"734": "police van, police wagon, paddy wagon, patrol wagon, wagon, black Maria",
"735": "poncho",
"736": "pool table, billiard table, snooker table",
"737": "pop bottle, soda bottle",
"738": "pot, flowerpot",
"739": "potter's wheel",
"740": "power drill",
"741": "prayer rug, prayer mat",
"742": "printer",
"743": "prison, prison house",
"744": "projectile, missile",
"745": "projector",
"746": "puck, hockey puck",
"747": "punching bag, punch bag, punching ball, punchball",
"748": "purse",
"749": "quill, quill pen",
"750": "quilt, comforter, comfort, puff",
"751": "racer, race car, racing car",
"752": "racket, racquet",
"753": "radiator",
"754": "radio, wireless",
"755": "radio telescope, radio reflector",
"756": "rain barrel",
"757": "recreational vehicle, RV, R.V.",
"758": "reel",
"759": "reflex camera",
"760": "refrigerator, icebox",
"761": "remote control, remote",
"762": "restaurant, eating house, eating place, eatery",
"763": "revolver, six-gun, six-shooter",
"764": "rifle",
"765": "rocking chair, rocker",
"766": "rotisserie",
"767": "rubber eraser, rubber, pencil eraser",
"768": "rugby ball",
"769": "rule, ruler",
"770": "running shoe",
"771": "safe",
"772": "safety pin",
"773": "saltshaker, salt shaker",
"774": "sandal",
"775": "sarong",
"776": "sax, saxophone",
"777": "scabbard",
"778": "scale, weighing machine",
"779": "school bus",
"780": "schooner",
"781": "scoreboard",
"782": "screen, CRT screen",
"783": "screw",
"784": "screwdriver",
"785": "seat belt, seatbelt",
"786": "sewing machine",
"787": "shield, buckler",
"788": "shoe shop, shoe-shop, shoe store",
"789": "shoji",
"790": "shopping basket",
"791": "shopping cart",
"792": "shovel",
"793": "shower cap",
"794": "shower curtain",
"795": "ski",
"796": "ski mask",
"797": "sleeping bag",
"798": "slide rule, slipstick",
"799": "sliding door",
"800": "slot, one-armed bandit",
"801": "snorkel",
"802": "snowmobile",
"803": "snowplow, snowplough",
"804": "soap dispenser",
"805": "soccer ball",
"806": "sock",
"807": "solar dish, solar collector, solar furnace",
"808": "sombrero",
"809": "soup bowl",
"810": "space bar",
"811": "space heater",
"812": "space shuttle",
"813": "spatula",
"814": "speedboat",
"815": "spider web, spider's web",
"816": "spindle",
"817": "sports car, sport car",
"818": "spotlight, spot",
"819": "stage",
"820": "steam locomotive",
"821": "steel arch bridge",
"822": "steel drum",
"823": "stethoscope",
"824": "stole",
"825": "stone wall",
"826": "stopwatch, stop watch",
"827": "stove",
"828": "strainer",
"829": "streetcar, tram, tramcar, trolley, trolley car",
"830": "stretcher",
"831": "studio couch, day bed",
"832": "stupa, tope",
"833": "submarine, pigboat, sub, U-boat",
"834": "suit, suit of clothes",
"835": "sundial",
"836": "sunglass",
"837": "sunglasses, dark glasses, shades",
"838": "sunscreen, sunblock, sun blocker",
"839": "suspension bridge",
"840": "swab, swob, mop",
"841": "sweatshirt",
"842": "swimming trunks, bathing trunks",
"843": "swing",
"844": "switch, electric switch, electrical switch",
"845": "syringe",
"846": "table lamp",
"847": "tank, army tank, armored combat vehicle, armoured combat vehicle",
"848": "tape player",
"849": "teapot",
"850": "teddy, teddy bear",
"851": "television, television system",
"852": "tennis ball",
"853": "thatch, thatched roof",
"854": "theater curtain, theatre curtain",
"855": "thimble",
"856": "thresher, thrasher, threshing machine",
"857": "throne",
"858": "tile roof",
"859": "toaster",
"860": "tobacco shop, tobacconist shop, tobacconist",
"861": "toilet seat",
"862": "torch",
"863": "totem pole",
"864": "tow truck, tow car, wrecker",
"865": "toyshop",
"866": "tractor",
"867": "trailer truck, tractor trailer, trucking rig, rig, articulated lorry, semi",
"868": "tray",
"869": "trench coat",
"870": "tricycle, trike, velocipede",
"871": "trimaran",
"872": "tripod",
"873": "triumphal arch",
"874": "trolleybus, trolley coach, trackless trolley",
"875": "trombone",
"876": "tub, vat",
"877": "turnstile",
"878": "typewriter keyboard",
"879": "umbrella",
"880": "unicycle, monocycle",
"881": "upright, upright piano",
"882": "vacuum, vacuum cleaner",
"883": "vase",
"884": "vault",
"885": "velvet",
"886": "vending machine",
"887": "vestment",
"888": "viaduct",
"889": "violin, fiddle",
"890": "volleyball",
"891": "waffle iron",
"892": "wall clock",
"893": "wallet, billfold, notecase, pocketbook",
"894": "wardrobe, closet, press",
"895": "warplane, military plane",
"896": "washbasin, handbasin, washbowl, lavabo, wash-hand basin",
"897": "washer, automatic washer, washing machine",
"898": "water bottle",
"899": "water jug",
"900": "water tower",
"901": "whiskey jug",
"902": "whistle",
"903": "wig",
"904": "window screen",
"905": "window shade",
"906": "Windsor tie",
"907": "wine bottle",
"908": "wing",
"909": "wok",
"910": "wooden spoon",
"911": "wool, woolen, woollen",
"912": "worm fence, snake fence, snake-rail fence, Virginia fence",
"913": "wreck",
"914": "yawl",
"915": "yurt",
"916": "web site, website, internet site, site",
"917": "comic book",
"918": "crossword puzzle, crossword",
"919": "street sign",
"920": "traffic light, traffic signal, stoplight",
"921": "book jacket, dust cover, dust jacket, dust wrapper",
"922": "menu",
"923": "plate",
"924": "guacamole",
"925": "consomme",
"926": "hot pot, hotpot",
"927": "trifle",
"928": "ice cream, icecream",
"929": "ice lolly, lolly, lollipop, popsicle",
"930": "French loaf",
"931": "bagel, beigel",
"932": "pretzel",
"933": "cheeseburger",
"934": "hotdog, hot dog, red hot",
"935": "mashed potato",
"936": "head cabbage",
"937": "broccoli",
"938": "cauliflower",
"939": "zucchini, courgette",
"940": "spaghetti squash",
"941": "acorn squash",
"942": "butternut squash",
"943": "cucumber, cuke",
"944": "artichoke, globe artichoke",
"945": "bell pepper",
"946": "cardoon",
"947": "mushroom",
"948": "Granny Smith",
"949": "strawberry",
"950": "orange",
"951": "lemon",
"952": "fig",
"953": "pineapple, ananas",
"954": "banana",
"955": "jackfruit, jak, jack",
"956": "custard apple",
"957": "pomegranate",
"958": "hay",
"959": "carbonara",
"960": "chocolate sauce, chocolate syrup",
"961": "dough",
"962": "meat loaf, meatloaf",
"963": "pizza, pizza pie",
"964": "potpie",
"965": "burrito",
"966": "red wine",
"967": "espresso",
"968": "cup",
"969": "eggnog",
"970": "alp",
"971": "bubble",
"972": "cliff, drop, drop-off",
"973": "coral reef",
"974": "geyser",
"975": "lakeside, lakeshore",
"976": "promontory, headland, head, foreland",
"977": "sandbar, sand bar",
"978": "seashore, coast, seacoast, sea-coast",
"979": "valley, vale",
"980": "volcano",
"981": "ballplayer, baseball player",
"982": "groom, bridegroom",
"983": "scuba diver",
"984": "rapeseed",
"985": "daisy",
"986": "yellow lady's slipper, yellow lady-slipper, Cypripedium calceolus, Cypripedium parviflorum",
"987": "corn",
"988": "acorn",
"989": "hip, rose hip, rosehip",
"990": "buckeye, horse chestnut, conker",
"991": "coral fungus",
"992": "agaric",
"993": "gyromitra",
"994": "stinkhorn, carrion fungus",
"995": "earthstar",
"996": "hen-of-the-woods, hen of the woods, Polyporus frondosus, Grifola frondosa",
"997": "bolete",
"998": "ear, spike, capitulum",
"999": "toilet tissue, toilet paper, bathroom tissue"
}

In [ ]:
pca_df["imagenet_name"] = pca_df["imagenet_label"].map({int(k): v for k, v in im_label_by_name.items()})

In [ ]:
pca_df

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
filtered_df = pca_df[pca_df.cluster_label == 17]

In [ ]:
snout_im_classes = filtered_df.groupby("imagenet_name").agg(
    samples_count=("imagenet_name", "count"),
    unique_images_count=("input_image_key", "nunique")
).sort_values("unique_images_count", ascending=False)
snout_im_classes

In [ ]:
snout_im_classes = snout_im_classes.reset_index()


In [ ]:
snout_im_classes = snout_im_classes[["unique_images_count", "samples_count", "imagenet_name"]]

In [ ]:
snout_im_classes.to_csv("pca_imagenet_class_counts_for_snouts.csv", index=False)

In [ ]:
! mv ./pca_imagenet_class_counts_for_snouts.csv  /Users/hariomnarang/Desktop/personal/blog/pages/posts/2026-06-19-disentangling-mixed4e-55/reports/pca-report/
